In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## PrePost Changedetection DataLoader

In [1]:
import os
import random
import numpy as np
import torch

from pathlib import Path
from collections import OrderedDict
from torch.utils.data import Dataset, DataLoader


# ============================================================
# GLOBAL REPRODUCIBILITY
# ============================================================

GLOBAL_SEED = 42


def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducibility settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(GLOBAL_SEED)


# ============================================================
# BASIC UTILITIES
# ============================================================

def normalize_image(image):
    """
    Convert uint8 image from [0, 255] to float32 [0, 1].

    This preserves the normalization used in the original
    STRATA implementation.
    """

    image = image.astype(np.float32) / 255.0
    return image


def ensure_channel(image):
    """
    Ensure that a grayscale image has a channel dimension.

    (H, W) -> (H, W, 1)
    """

    if image.ndim == 2:
        image = image[:, :, None]

    return image


def safe_copy(array):
    """
    Make the array contiguous after flips and rotations.
    """

    return np.ascontiguousarray(array)


def get_event_name(file_path):
    """
    Extract event name from an NPZ filename.

    Examples:
        EARTHQUAKE-TURKEY_000013_0_0.npz
            -> EARTHQUAKE-TURKEY

        TEXAS-TORNADOES_000006_256_512.npz
            -> TEXAS-TORNADOES
    """

    filename = os.path.basename(file_path)
    return filename.split("_")[0]

In [2]:
def joint_random_flip(
    pre_image,
    post_image,
    damage_mask,
):
    """
    Apply identical horizontal and vertical flips to
    pre-image, post-image, and damage mask.
    """

    # Horizontal flip
    if random.random() < 0.5:
        pre_image = pre_image[:, ::-1, :]
        post_image = post_image[:, ::-1, :]
        damage_mask = damage_mask[:, ::-1]

    # Vertical flip
    if random.random() < 0.5:
        pre_image = pre_image[::-1, :, :]
        post_image = post_image[::-1, :, :]
        damage_mask = damage_mask[::-1, :]

    return pre_image, post_image, damage_mask


def joint_random_rotate(
    pre_image,
    post_image,
    damage_mask,
):
    """
    Apply the same random 0/90/180/270-degree rotation
    to both temporal images and the label.
    """

    k = random.randint(0, 3)

    pre_image = np.rot90(
        pre_image,
        k=k,
        axes=(0, 1),
    )

    post_image = np.rot90(
        post_image,
        k=k,
        axes=(0, 1),
    )

    damage_mask = np.rot90(
        damage_mask,
        k=k,
        axes=(0, 1),
    )

    return pre_image, post_image, damage_mask

In [3]:
def paired_brightness_scaling(
    pre_image,
    post_image,
    probability=0.5,
    minimum_factor=0.8,
    maximum_factor=1.2,
):
    """
    Apply the same brightness factor to pre and post images.

    Using the same factor avoids creating artificial
    temporal differences.
    """

    if random.random() < probability:

        factor = random.uniform(
            minimum_factor,
            maximum_factor,
        )

        pre_image = pre_image * factor
        post_image = post_image * factor

    pre_image = np.clip(
        pre_image,
        0.0,
        1.0,
    )

    post_image = np.clip(
        post_image,
        0.0,
        1.0,
    )

    return pre_image, post_image


def paired_gaussian_noise(
    pre_image,
    post_image,
    probability=0.3,
    noise_std=0.02,
):
    """
    Apply mild Gaussian noise to both temporal images.

    The same noise field is used for both images to avoid
    creating artificial change evidence.
    """

    if random.random() < probability:

        noise = np.random.normal(
            loc=0.0,
            scale=noise_std,
            size=pre_image.shape,
        ).astype(np.float32)

        pre_image = pre_image + noise
        post_image = post_image + noise

    pre_image = np.clip(
        pre_image,
        0.0,
        1.0,
    )

    post_image = np.clip(
        post_image,
        0.0,
        1.0,
    )

    return pre_image, post_image

In [4]:
class NPZCache:
    """
    Small per-worker in-memory cache.

    A limited cache is used because each paired sample contains
    two RGB images and one mask.
    """

    def __init__(self, max_size=128):

        self.cache = OrderedDict()
        self.max_size = max_size

    def get(self, file_path):

        file_path = str(file_path)

        if file_path in self.cache:
            self.cache.move_to_end(file_path)
            return self.cache[file_path]

        # The context manager closes the NPZ file immediately
        # after its arrays are copied.
        with np.load(file_path) as data:

            sample = {
                "pre_image": data["pre_image"].copy(),
                "post_image": data["post_image"].copy(),
                "damage_mask": data["damage_mask"].copy(),
            }

        if len(self.cache) >= self.max_size:
            self.cache.popitem(last=False)

        self.cache[file_path] = sample

        return sample

In [5]:
class PairedBuildingDamageDataset(Dataset):
    """
    Supervised paired pre/post dataset for bi-temporal
    building-damage segmentation.

    Each item returns:
        pre_image:   FloatTensor [3, H, W]
        post_image:  FloatTensor [3, H, W]
        damage_mask: LongTensor  [H, W]
        valid_mask:  BoolTensor  [H, W]
        event_name:  str

    Label definition:
        0 = Background
        1 = Intact
        2 = Damage
        3 = Destroyed
    """

    def __init__(
        self,
        root_dir=None,
        files=None,
        augment=False,
        cache_size=128,
    ):

        # ----------------------------------------------------
        # Use an explicitly supplied split whenever available
        # ----------------------------------------------------

        if files is not None:

            self.files = sorted([
                str(file_path)
                for file_path in files
            ])

        elif root_dir is not None:

            root_dir = Path(root_dir)

            self.files = sorted([
                str(file_path)
                for file_path in root_dir.glob("*.npz")
            ])

        else:
            raise ValueError(
                "Either root_dir or files must be provided."
            )

        if len(self.files) == 0:
            raise RuntimeError(
                "No NPZ files were found for this dataset."
            )

        self.augment = augment
        self.cache = NPZCache(
            max_size=cache_size
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):

        file_path = self.files[index]
        sample = self.cache.get(file_path)

        pre_image = sample["pre_image"].copy()
        post_image = sample["post_image"].copy()
        damage_mask = sample["damage_mask"].copy()

        # ----------------------------------------------------
        # Basic validation
        # ----------------------------------------------------

        pre_image = ensure_channel(pre_image)
        post_image = ensure_channel(post_image)

        if pre_image.shape != post_image.shape:
            raise ValueError(
                f"Pre/post shape mismatch in {file_path}: "
                f"{pre_image.shape} versus {post_image.shape}"
            )

        if pre_image.shape[:2] != damage_mask.shape:
            raise ValueError(
                f"Image/mask shape mismatch in {file_path}: "
                f"{pre_image.shape[:2]} versus "
                f"{damage_mask.shape}"
            )

        unique_labels = set(
            np.unique(damage_mask).tolist()
        )

        if not unique_labels.issubset({0, 1, 2, 3}):
            raise ValueError(
                f"Unexpected labels in {file_path}: "
                f"{sorted(unique_labels)}"
            )

        # ----------------------------------------------------
        # Normalization: same [0,1] convention as STRATA
        # ----------------------------------------------------

        pre_image = normalize_image(pre_image)
        post_image = normalize_image(post_image)

        # ----------------------------------------------------
        # Joint paired augmentation
        # ----------------------------------------------------

        if self.augment:

            pre_image, post_image, damage_mask = (
                joint_random_flip(
                    pre_image,
                    post_image,
                    damage_mask,
                )
            )

            pre_image, post_image, damage_mask = (
                joint_random_rotate(
                    pre_image,
                    post_image,
                    damage_mask,
                )
            )

            # Use stronger appearance augmentation for patches
            # containing Damage or Destroyed buildings.
            contains_rare_damage = np.any(
                (damage_mask == 2) |
                (damage_mask == 3)
            )

            if contains_rare_damage:

                pre_image, post_image = (
                    paired_gaussian_noise(
                        pre_image,
                        post_image,
                        probability=0.3,
                        noise_std=0.02,
                    )
                )

                pre_image, post_image = (
                    paired_brightness_scaling(
                        pre_image,
                        post_image,
                        probability=0.5,
                    )
                )

            else:

                pre_image, post_image = (
                    paired_gaussian_noise(
                        pre_image,
                        post_image,
                        probability=0.15,
                        noise_std=0.02,
                    )
                )

                pre_image, post_image = (
                    paired_brightness_scaling(
                        pre_image,
                        post_image,
                        probability=0.3,
                    )
                )

        # ----------------------------------------------------
        # Building-only valid mask
        #
        # A separate building mask is unnecessary because:
        #   0 = background
        #   1, 2, 3 = building damage classes
        # ----------------------------------------------------

        valid_mask = damage_mask > 0

        # ----------------------------------------------------
        # Safe contiguous arrays after flip/rotation
        # ----------------------------------------------------

        pre_image = safe_copy(pre_image)
        post_image = safe_copy(post_image)
        damage_mask = safe_copy(damage_mask)
        valid_mask = safe_copy(valid_mask)

        # ----------------------------------------------------
        # Convert HWC -> CHW
        # ----------------------------------------------------

        pre_image = np.transpose(
            pre_image,
            (2, 0, 1),
        )

        post_image = np.transpose(
            post_image,
            (2, 0, 1),
        )

        # ----------------------------------------------------
        # Convert to tensors
        # ----------------------------------------------------

        pre_image = torch.from_numpy(
            pre_image
        ).float()

        post_image = torch.from_numpy(
            post_image
        ).float()

        damage_mask = torch.from_numpy(
            damage_mask
        ).long()

        valid_mask = torch.from_numpy(
            valid_mask
        ).bool()

        event_name = get_event_name(
            file_path
        )

        return (
            pre_image,
            post_image,
            damage_mask,
            valid_mask,
            event_name,
        )

In [6]:
def seed_dataloader_worker(worker_id):
    """
    Give every DataLoader worker a deterministic but distinct seed.
    """

    worker_seed = (
        torch.initial_seed() % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_paired_loader(
    dataset,
    batch_size=16,
    shuffle=True,
    sampler=None,
    drop_last=False,
    num_workers=4,
    pin_memory=True,
):

    # A sampler and shuffle=True cannot be used together.
    if sampler is not None:
        shuffle = False

    generator = torch.Generator()
    generator.manual_seed(GLOBAL_SEED)

    loader_arguments = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "sampler": sampler,
        "drop_last": drop_last,
        "num_workers": num_workers,
        "pin_memory": pin_memory,
        "worker_init_fn": seed_dataloader_worker,
        "generator": generator,
    }

    # These arguments are only valid when workers are enabled.
    if num_workers > 0:
        loader_arguments.update({
            "persistent_workers": True,
            "prefetch_factor": 2,
        })

    return DataLoader(**loader_arguments)

In [7]:
DATA_ROOT = (
    "/content/ebd_target_prepost_npz"
)

In [8]:
import os
import zipfile

ZIP_PATH = (
    "/content/drive/MyDrive/"
    "EBD_Target_PrePost_256.zip"
)

DATA_ROOT = (
    "/content/ebd_target_prepost_npz"
)

os.makedirs(
    DATA_ROOT,
    exist_ok=True,
)

extraction_marker = os.path.join(
    DATA_ROOT,
    ".extraction_complete",
)

if not os.path.exists(extraction_marker):

    print("Extracting paired dataset...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as zip_ref:
        zip_ref.extractall(DATA_ROOT)

    Path(extraction_marker).touch()

    print("Extraction completed.")

else:
    print("Dataset was already extracted.")

Extracting paired dataset...
Extraction completed.


In [8]:
full_dataset = PairedBuildingDamageDataset(
    root_dir=DATA_ROOT,
    augment=False,
    cache_size=32,
)

print("Dataset size:", len(full_dataset))

sample = full_dataset[0]

pre_image, post_image, damage_mask, valid_mask, event_name = sample

print("Event:", event_name)
print("Pre image:", pre_image.shape, pre_image.dtype)
print("Post image:", post_image.shape, post_image.dtype)
print("Damage mask:", damage_mask.shape, damage_mask.dtype)
print("Valid mask:", valid_mask.shape, valid_mask.dtype)

print(
    "Damage classes:",
    torch.unique(damage_mask).tolist(),
)

print(
    "Valid building pixels:",
    valid_mask.sum().item(),
)

print(
    "Pre range:",
    pre_image.min().item(),
    pre_image.max().item(),
)

print(
    "Post range:",
    post_image.min().item(),
    post_image.max().item(),
)

Dataset size: 13105
Event: EARTHQUAKE-TURKEY
Pre image: torch.Size([3, 256, 256]) torch.float32
Post image: torch.Size([3, 256, 256]) torch.float32
Damage mask: torch.Size([256, 256]) torch.int64
Valid mask: torch.Size([256, 256]) torch.bool
Damage classes: [0, 1]
Valid building pixels: 1936
Pre range: 0.0235294122248888 1.0
Post range: 0.0 1.0


## Model Architecture

In [9]:
from collections import OrderedDict

import torch
import torch.nn as nn


class SiameseUNet(nn.Module):
    """
    Siamese U-Net adapted from the Microsoft Building Damage
    Assessment implementation.

    Inputs
    ------
    pre_image:
        Tensor [B, 3, H, W]

    post_image:
        Tensor [B, 3, H, W]

    Outputs
    -------
    pre_building_logits:
        Tensor [B, 2, H, W]

    post_building_logits:
        Tensor [B, 2, H, W]

    damage_logits:
        Tensor [B, 4, H, W]

    Damage classes
    --------------
        0 = Background
        1 = Intact
        2 = Damage
        3 = Destroyed
    """

    def __init__(
        self,
        in_channels=3,
        building_classes=2,
        damage_classes=4,
        init_features=16,
    ):
        super().__init__()

        features = init_features

        # ====================================================
        # Shared U-Net encoder
        # ====================================================

        self.encoder1 = self._block(
            in_channels,
            features,
            name="enc1_",
        )

        self.pool1 = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

        self.encoder2 = self._block(
            features,
            features * 2,
            name="enc2_",
        )

        self.pool2 = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

        self.encoder3 = self._block(
            features * 2,
            features * 4,
            name="enc3_",
        )

        self.pool3 = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

        self.encoder4 = self._block(
            features * 4,
            features * 8,
            name="enc4_",
        )

        self.pool4 = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )

        self.bottleneck = self._block(
            features * 8,
            features * 16,
            name="bottleneck_",
        )

        # ====================================================
        # Shared building-localization decoder
        #
        # This decoder is called once for the pre-event branch
        # and once for the post-event branch. Its parameters
        # are shared across both temporal inputs.
        # ====================================================

        self.upconv4 = nn.ConvTranspose2d(
            features * 16,
            features * 8,
            kernel_size=2,
            stride=2,
        )

        self.decoder4 = self._block(
            features * 16,
            features * 8,
            name="dec4_",
        )

        self.upconv3 = nn.ConvTranspose2d(
            features * 8,
            features * 4,
            kernel_size=2,
            stride=2,
        )

        self.decoder3 = self._block(
            features * 8,
            features * 4,
            name="dec3_",
        )

        self.upconv2 = nn.ConvTranspose2d(
            features * 4,
            features * 2,
            kernel_size=2,
            stride=2,
        )

        self.decoder2 = self._block(
            features * 4,
            features * 2,
            name="dec2_",
        )

        self.upconv1 = nn.ConvTranspose2d(
            features * 2,
            features,
            kernel_size=2,
            stride=2,
        )

        self.decoder1 = self._block(
            features * 2,
            features,
            name="dec1_",
        )

        self.building_classifier = nn.Conv2d(
            in_channels=features,
            out_channels=building_classes,
            kernel_size=1,
        )

        # ====================================================
        # Siamese damage decoder
        #
        # Directional feature difference:
        #     post_features - pre_features
        #
        # This is preserved from the Microsoft implementation.
        # ====================================================

        self.damage_upconv4 = nn.ConvTranspose2d(
            features * 16,
            features * 8,
            kernel_size=2,
            stride=2,
        )

        self.damage_decoder4 = self._block(
            features * 16,
            features * 16,
            name="damage_dec4_",
        )

        self.damage_upconv3 = nn.ConvTranspose2d(
            features * 16,
            features * 4,
            kernel_size=2,
            stride=2,
        )

        self.damage_decoder3 = self._block(
            features * 8,
            features * 8,
            name="damage_dec3_",
        )

        self.damage_upconv2 = nn.ConvTranspose2d(
            features * 8,
            features * 2,
            kernel_size=2,
            stride=2,
        )

        self.damage_decoder2 = self._block(
            features * 4,
            features * 4,
            name="damage_dec2_",
        )

        self.damage_upconv1 = nn.ConvTranspose2d(
            features * 4,
            features,
            kernel_size=2,
            stride=2,
        )

        self.damage_decoder1 = self._block(
            features * 2,
            features * 2,
            name="damage_dec1_",
        )

        self.damage_classifier = nn.Conv2d(
            in_channels=features * 2,
            out_channels=damage_classes,
            kernel_size=1,
        )

    # ========================================================
    # Shared encoder
    # ========================================================

    def encode(self, image):

        enc1 = self.encoder1(image)

        enc2 = self.encoder2(
            self.pool1(enc1)
        )

        enc3 = self.encoder3(
            self.pool2(enc2)
        )

        enc4 = self.encoder4(
            self.pool3(enc3)
        )

        bottleneck = self.bottleneck(
            self.pool4(enc4)
        )

        return {
            "enc1": enc1,
            "enc2": enc2,
            "enc3": enc3,
            "enc4": enc4,
            "bottleneck": bottleneck,
        }

    # ========================================================
    # Shared localization decoder
    # ========================================================

    def decode_building(self, features):

        dec4 = self.upconv4(
            features["bottleneck"]
        )

        dec4 = torch.cat(
            [dec4, features["enc4"]],
            dim=1,
        )

        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)

        dec3 = torch.cat(
            [dec3, features["enc3"]],
            dim=1,
        )

        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)

        dec2 = torch.cat(
            [dec2, features["enc2"]],
            dim=1,
        )

        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)

        dec1 = torch.cat(
            [dec1, features["enc1"]],
            dim=1,
        )

        dec1 = self.decoder1(dec1)

        building_logits = self.building_classifier(
            dec1
        )

        return building_logits

    # ========================================================
    # Siamese damage decoder
    # ========================================================

    def decode_damage(
        self,
        pre_features,
        post_features,
    ):

        # Bottleneck difference
        damage = (
            post_features["bottleneck"]
            - pre_features["bottleneck"]
        )

        # ----------------------------------------------------
        # Decoder level 4
        # ----------------------------------------------------

        damage = self.damage_upconv4(
            damage
        )

        enc4_difference = (
            post_features["enc4"]
            - pre_features["enc4"]
        )

        damage = torch.cat(
            [enc4_difference, damage],
            dim=1,
        )

        damage = self.damage_decoder4(
            damage
        )

        # ----------------------------------------------------
        # Decoder level 3
        # ----------------------------------------------------

        damage = self.damage_upconv3(
            damage
        )

        enc3_difference = (
            post_features["enc3"]
            - pre_features["enc3"]
        )

        damage = torch.cat(
            [enc3_difference, damage],
            dim=1,
        )

        damage = self.damage_decoder3(
            damage
        )

        # ----------------------------------------------------
        # Decoder level 2
        # ----------------------------------------------------

        damage = self.damage_upconv2(
            damage
        )

        enc2_difference = (
            post_features["enc2"]
            - pre_features["enc2"]
        )

        damage = torch.cat(
            [enc2_difference, damage],
            dim=1,
        )

        damage = self.damage_decoder2(
            damage
        )

        # ----------------------------------------------------
        # Decoder level 1
        # ----------------------------------------------------

        damage = self.damage_upconv1(
            damage
        )

        enc1_difference = (
            post_features["enc1"]
            - pre_features["enc1"]
        )

        damage = torch.cat(
            [enc1_difference, damage],
            dim=1,
        )

        damage = self.damage_decoder1(
            damage
        )

        damage_logits = self.damage_classifier(
            damage
        )

        return damage_logits

    # ========================================================
    # Complete forward pass
    # ========================================================

    def forward(
        self,
        pre_image,
        post_image,
    ):

        if pre_image.shape != post_image.shape:
            raise ValueError(
                "Pre/post input shapes must match. "
                f"Received {pre_image.shape} and "
                f"{post_image.shape}."
            )

        # Shared-weight temporal encoding
        pre_features = self.encode(
            pre_image
        )

        post_features = self.encode(
            post_image
        )

        # Shared building localization
        pre_building_logits = (
            self.decode_building(
                pre_features
            )
        )

        post_building_logits = (
            self.decode_building(
                post_features
            )
        )

        # Multi-scale temporal damage classification
        damage_logits = self.decode_damage(
            pre_features,
            post_features,
        )

        return (
            pre_building_logits,
            post_building_logits,
            damage_logits,
        )

    # ========================================================
    # Convolution block
    # ========================================================

    @staticmethod
    def _block(
        in_channels,
        out_channels,
        name,
    ):

        return nn.Sequential(
            OrderedDict([
                (
                    name + "conv1",
                    nn.Conv2d(
                        in_channels=in_channels,
                        out_channels=out_channels,
                        kernel_size=3,
                        padding=1,
                        bias=False,
                    ),
                ),
                (
                    name + "norm1",
                    nn.BatchNorm2d(
                        num_features=out_channels
                    ),
                ),
                (
                    name + "relu1",
                    nn.ReLU(inplace=True),
                ),
                (
                    name + "conv2",
                    nn.Conv2d(
                        in_channels=out_channels,
                        out_channels=out_channels,
                        kernel_size=3,
                        padding=1,
                        bias=False,
                    ),
                ),
                (
                    name + "norm2",
                    nn.BatchNorm2d(
                        num_features=out_channels
                    ),
                ),
                (
                    name + "relu2",
                    nn.ReLU(inplace=True),
                ),
            ])
        )

In [10]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = SiameseUNet(
    in_channels=3,
    building_classes=2,
    damage_classes=4,
    init_features=16,
).to(device)


dummy_pre = torch.randn(
    2,
    3,
    256,
    256,
    device=device,
)

dummy_post = torch.randn(
    2,
    3,
    256,
    256,
    device=device,
)


model.eval()

with torch.inference_mode():

    (
        pre_building_logits,
        post_building_logits,
        damage_logits,
    ) = model(
        dummy_pre,
        dummy_post,
    )


print("Device:", device)

print(
    "Pre-building output:",
    pre_building_logits.shape,
)

print(
    "Post-building output:",
    post_building_logits.shape,
)

print(
    "Damage output:",
    damage_logits.shape,
)

Device: cuda
Pre-building output: torch.Size([2, 2, 256, 256])
Post-building output: torch.Size([2, 2, 256, 256])
Damage output: torch.Size([2, 4, 256, 256])


In [11]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

model_size_mb = (
    total_parameters * 4 / (1024**2)
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

print(
    f"Approximate FP32 parameter size: "
    f"{model_size_mb:.2f} MB"
)

Total parameters: 3,728,694
Trainable parameters: 3,728,694
Approximate FP32 parameter size: 14.22 MB


In [12]:
sample_loader = get_paired_loader(
    dataset=full_dataset,
    batch_size=2,
    shuffle=False,
    drop_last=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

(
    pre_image,
    post_image,
    damage_mask,
    valid_mask,
    event_name,
) = next(iter(sample_loader))


pre_image = pre_image.to(
    device,
    non_blocking=True,
)

post_image = post_image.to(
    device,
    non_blocking=True,
)


model.eval()

with torch.inference_mode():

    (
        pre_building_logits,
        post_building_logits,
        damage_logits,
    ) = model(
        pre_image,
        post_image,
    )


print("Events:", list(event_name))

print("Input pre:", pre_image.shape)
print("Input post:", post_image.shape)

print(
    "Pre-building logits:",
    pre_building_logits.shape,
)

print(
    "Post-building logits:",
    post_building_logits.shape,
)

print(
    "Damage logits:",
    damage_logits.shape,
)

print(
    "Reference classes:",
    torch.unique(damage_mask).tolist(),
)

print(
    "Predicted classes before training:",
    torch.unique(
        damage_logits.argmax(dim=1).cpu()
    ).tolist(),
)

Events: ['EARTHQUAKE-TURKEY', 'EARTHQUAKE-TURKEY']
Input pre: torch.Size([2, 3, 256, 256])
Input post: torch.Size([2, 3, 256, 256])
Pre-building logits: torch.Size([2, 2, 256, 256])
Post-building logits: torch.Size([2, 2, 256, 256])
Damage logits: torch.Size([2, 4, 256, 256])
Reference classes: [0, 1]
Predicted classes before training: [3]


## Loss & Training

In [13]:
# ============================================================
# MICROSOFT SIAMESE U-NET CONFIGURATION
# ============================================================

from pathlib import Path
import torch


# ------------------------------------------------------------
# Data and existing STRATA split
# ------------------------------------------------------------

PAIRED_DATA_ROOT = Path(
    "/content/ebd_target_prepost_npz"
)

EXPERIMENT_SPLIT_PATH = Path(
    "/content/drive/MyDrive/STRATA_runs/splits/"
    "ebd_npz_event_EQTURKEY_TEXASTORNADO_"
    "labeled_unlabeled_500.npz"
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "bitemporal_baselines/Siamese_UNet_500"
)

LOCALIZATION_MODEL_DIR = (
    OUTPUT_DIR / "localization_models"
)

DAMAGE_MODEL_DIR = (
    OUTPUT_DIR / "damage_models"
)

BEST_LOCALIZATION_PATH = (
    LOCALIZATION_MODEL_DIR /
    "best_localization_model.pth"
)

BEST_DAMAGE_PATH = (
    DAMAGE_MODEL_DIR /
    "best_damage_model.pth"
)

FINAL_EXCEL_PATH = (
    OUTPUT_DIR /
    "final_validation_global_and_event_metrics.xlsx"
)

HISTORY_PATH = (
    OUTPUT_DIR /
    "training_history.pth"
)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

NUM_BUILDING_CLASSES = 2
NUM_DAMAGE_CLASSES = 4
INITIAL_FEATURES = 16


# ------------------------------------------------------------
# Two-stage training
# ------------------------------------------------------------

LOCALIZATION_EPOCHS = 30
DAMAGE_EPOCHS = 100

EVAL_EVERY = 3
EARLY_STOP_PATIENCE = 5

TRAIN_BATCH_SIZE = 64
VAL_BATCH_SIZE = 32

NUM_WORKERS = 0
GRADIENT_ACCUMULATION_STEPS = 2
MAX_GRAD_NORM = 5.0

PIN_MEMORY = torch.cuda.is_available()
USE_AMP = torch.cuda.is_available()


# ------------------------------------------------------------
# Optimizers
# ------------------------------------------------------------

LOCALIZATION_LEARNING_RATE = 1e-3
DAMAGE_LEARNING_RATE = 5e-4

MIN_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 0.0


# ------------------------------------------------------------
# Microsoft class weights
# ------------------------------------------------------------

# Official Microsoft building weights
BUILDING_CLASS_WEIGHTS = [
    1.0,   # Background
    15.0,  # Building
]

# Adapted from Microsoft's five-class damage weights:
# [1, 35, 70, 150, 120]
#
# No-damage and Minor are merged into Intact in STRATA.
DAMAGE_CLASS_WEIGHTS = [
    1.0,    # Background
    35.0,   # Intact
    150.0,  # Damage
    120.0,  # Destroyed
]


# ------------------------------------------------------------
# Stage-specific loss weights
# ------------------------------------------------------------

LOCALIZATION_LOSS_WEIGHTS = [
    0.5,  # Pre-event building loss
    0.5,  # Post-event building loss
    0.0,  # Damage loss
]

DAMAGE_LOSS_WEIGHTS = [
    0.0,  # Pre-event building loss
    0.0,  # Post-event building loss
    1.0,  # Damage loss
]


# ------------------------------------------------------------
# Sampling
# ------------------------------------------------------------

USE_WEIGHTED_SAMPLER = True

INTACT_SAMPLE_WEIGHT = 1.0
DAMAGE_SAMPLE_WEIGHT = 5.0
DESTROYED_SAMPLE_WEIGHT = 6.0


# ------------------------------------------------------------
# Reproducibility and device
# ------------------------------------------------------------

SEED = 42

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("Device:", DEVICE)
print("Data:", PAIRED_DATA_ROOT)
print("Split:", EXPERIMENT_SPLIT_PATH)
print("Output:", OUTPUT_DIR)

Device: cuda
Data: /content/ebd_target_prepost_npz
Split: /content/drive/MyDrive/STRATA_runs/splits/ebd_npz_event_EQTURKEY_TEXASTORNADO_labeled_unlabeled_500.npz
Output: /content/drive/MyDrive/STRATA_runs/bitemporal_baselines/Siamese_UNet_500


In [14]:
# ============================================================
# MICROSOFT SIAMESE U-NET:
# SPLITS, INTRINSIC LOSS, TRAINING, AND EVALUATION
# ============================================================

import os
import gc
import random

from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.amp import autocast, GradScaler
from torch.utils.data import WeightedRandomSampler
from tqdm.auto import tqdm


# ============================================================
# SETUP
# ============================================================

def set_experiment_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_experiment_seed(SEED)

for directory in [
    OUTPUT_DIR,
    LOCALIZATION_MODEL_DIR,
    DAMAGE_MODEL_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# LOAD EXISTING STRATA SPLIT
# ============================================================

def map_split_paths(
    old_paths,
    new_data_root,
    split_name,
):

    mapped_paths = []
    missing_paths = []

    for old_path in old_paths:

        filename = os.path.basename(
            str(old_path)
        )

        new_path = (
            Path(new_data_root) /
            filename
        )

        if new_path.exists():
            mapped_paths.append(
                str(new_path)
            )
        else:
            missing_paths.append(
                str(new_path)
            )

    if missing_paths:

        print(
            f"Missing {split_name} files: "
            f"{len(missing_paths)}"
        )

        for path in missing_paths[:20]:
            print(path)

        raise FileNotFoundError(
            f"Existing {split_name} split "
            f"could not be mapped completely."
        )

    if len(mapped_paths) != len(
        set(mapped_paths)
    ):
        raise RuntimeError(
            f"Duplicate files detected in "
            f"{split_name} split."
        )

    return sorted(mapped_paths)


def load_siamese_split():

    if not EXPERIMENT_SPLIT_PATH.exists():
        raise FileNotFoundError(
            f"Split not found: "
            f"{EXPERIMENT_SPLIT_PATH}"
        )

    with np.load(
        EXPERIMENT_SPLIT_PATH,
        allow_pickle=True,
    ) as split_data:

        required_keys = {
            "labeled",
            "val",
        }

        if not required_keys.issubset(
            set(split_data.files)
        ):
            raise KeyError(
                f"Required keys: {required_keys}; "
                f"available keys: "
                f"{set(split_data.files)}"
            )

        old_train_files = list(
            split_data["labeled"]
        )

        old_val_files = list(
            split_data["val"]
        )

    train_files = map_split_paths(
        old_train_files,
        PAIRED_DATA_ROOT,
        "training",
    )

    val_files = map_split_paths(
        old_val_files,
        PAIRED_DATA_ROOT,
        "validation",
    )

    overlap = (
        set(train_files) &
        set(val_files)
    )

    if overlap:
        raise RuntimeError(
            f"Train/validation leakage: "
            f"{len(overlap)} files."
        )

    return train_files, val_files


# ============================================================
# SPLIT INSPECTION
# ============================================================

def inspect_split(
    files,
    split_name,
):

    event_counts = Counter()
    class_counts = Counter()

    for file_path in tqdm(
        files,
        desc=f"Inspecting {split_name}",
        leave=False,
    ):

        event_counts[
            get_event_name(file_path)
        ] += 1

        with np.load(file_path) as data:
            labels = np.unique(
                data["damage_mask"]
            )

        for label in labels:
            if label != 0:
                class_counts[
                    int(label)
                ] += 1

    print(
        f"\n===== {split_name.upper()} ====="
    )

    print("Total files:", len(files))

    print("Events:")

    for event, count in sorted(
        event_counts.items()
    ):
        print(f"  {event}: {count:,}")

    print("Files containing classes:")
    print(
        f"  Intact: "
        f"{class_counts[1]:,}"
    )
    print(
        f"  Damage: "
        f"{class_counts[2]:,}"
    )
    print(
        f"  Destroyed: "
        f"{class_counts[3]:,}"
    )


# ============================================================
# WEIGHTED SAMPLER
# ============================================================

def compute_sample_weights(files):

    weights = []

    for file_path in tqdm(
        files,
        desc="Computing sample weights",
        leave=False,
    ):

        with np.load(file_path) as data:
            labels = set(
                np.unique(
                    data["damage_mask"]
                ).tolist()
            )

        weight = INTACT_SAMPLE_WEIGHT

        if 2 in labels:
            weight = max(
                weight,
                DAMAGE_SAMPLE_WEIGHT,
            )

        if 3 in labels:
            weight = max(
                weight,
                DESTROYED_SAMPLE_WEIGHT,
            )

        weights.append(weight)

    return torch.tensor(
        weights,
        dtype=torch.double,
    )


def build_dataloaders():

    train_files, val_files = (
        load_siamese_split()
    )

    inspect_split(
        train_files,
        "training",
    )

    inspect_split(
        val_files,
        "validation",
    )

    train_dataset = (
        PairedBuildingDamageDataset(
            files=train_files,
            augment=True,
            cache_size=128,
        )
    )

    val_dataset = (
        PairedBuildingDamageDataset(
            files=val_files,
            augment=False,
            cache_size=128,
        )
    )

    if USE_WEIGHTED_SAMPLER:

        sample_weights = (
            compute_sample_weights(
                train_files
            )
        )

        sampler_generator = (
            torch.Generator()
        )

        sampler_generator.manual_seed(
            SEED
        )

        train_sampler = (
            WeightedRandomSampler(
                weights=sample_weights,
                num_samples=len(
                    sample_weights
                ),
                replacement=True,
                generator=(
                    sampler_generator
                ),
            )
        )

        train_loader = get_paired_loader(
            dataset=train_dataset,
            batch_size=(
                TRAIN_BATCH_SIZE
            ),
            shuffle=False,
            sampler=train_sampler,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

    else:

        train_loader = get_paired_loader(
            dataset=train_dataset,
            batch_size=(
                TRAIN_BATCH_SIZE
            ),
            shuffle=True,
            sampler=None,
            drop_last=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

    val_loader = get_paired_loader(
        dataset=val_dataset,
        batch_size=VAL_BATCH_SIZE,
        shuffle=False,
        sampler=None,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    print("\n===== SPLIT CHECK =====")
    print("Train:", len(train_files))
    print("Validation:", len(val_files))
    print(
        "Overlap:",
        len(
            set(train_files) &
            set(val_files)
        ),
    )

    return (
        train_loader,
        val_loader,
        train_files,
        val_files,
    )


# ============================================================
# MICROSOFT INTRINSIC LOSSES
# ============================================================

class MicrosoftSiameseLoss(nn.Module):
    """
    Original three-component weighted Cross-Entropy objective:

        L = w1 * CE(pre building)
          + w2 * CE(post building)
          + w3 * CE(damage)
    """

    def __init__(
        self,
        building_class_weights,
        damage_class_weights,
    ):

        super().__init__()

        self.register_buffer(
            "building_weights",
            torch.tensor(
                building_class_weights,
                dtype=torch.float32,
            ),
        )

        self.register_buffer(
            "damage_weights",
            torch.tensor(
                damage_class_weights,
                dtype=torch.float32,
            ),
        )

        self.pre_building_criterion = (
            nn.CrossEntropyLoss(
                weight=self.building_weights
            )
        )

        self.post_building_criterion = (
            nn.CrossEntropyLoss(
                weight=self.building_weights
            )
        )

        self.damage_criterion = (
            nn.CrossEntropyLoss(
                weight=self.damage_weights
            )
        )

    def forward(
        self,
        pre_building_logits,
        post_building_logits,
        damage_logits,
        building_target,
        damage_target,
        loss_weights,
    ):

        pre_building_loss = (
            self.pre_building_criterion(
                pre_building_logits,
                building_target,
            )
        )

        post_building_loss = (
            self.post_building_criterion(
                post_building_logits,
                building_target,
            )
        )

        damage_loss = (
            self.damage_criterion(
                damage_logits,
                damage_target,
            )
        )

        total_loss = (
            loss_weights[0] *
            pre_building_loss
            +
            loss_weights[1] *
            post_building_loss
            +
            loss_weights[2] *
            damage_loss
        )

        return {
            "total": total_loss,
            "pre_building": (
                pre_building_loss
            ),
            "post_building": (
                post_building_loss
            ),
            "damage": damage_loss,
        }


# ============================================================
# ORIGINAL MICROSOFT DAMAGE GATING
# ============================================================

def apply_pre_building_gate(
    damage_logits,
    pre_building_logits,
):
    """
    Preserve Microsoft's original inference/training behavior.

    The predicted pre-event building mask is applied to every
    damage logit channel. Outside predicted buildings all logits
    become zero, and argmax consequently returns Background.
    """

    with torch.no_grad():

        predicted_pre_building = (
            torch.argmax(
                pre_building_logits,
                dim=1,
            )
        ).float()

    return (
        damage_logits *
        predicted_pre_building.unsqueeze(1)
    )


# ============================================================
# FREEZE FOR DAMAGE PHASE
# ============================================================

def freeze_localization_components(
    model,
):
    """
    Freeze shared encoder and building-localization decoder,
    following the Microsoft damage-training stage.
    """

    modules_to_freeze = [
        model.encoder1,
        model.encoder2,
        model.encoder3,
        model.encoder4,
        model.bottleneck,

        model.upconv4,
        model.decoder4,
        model.upconv3,
        model.decoder3,
        model.upconv2,
        model.decoder2,
        model.upconv1,
        model.decoder1,

        model.building_classifier,
    ]

    for module in modules_to_freeze:

        for parameter in module.parameters():
            parameter.requires_grad = False

    for parameter in model.parameters():

        if parameter.requires_grad:
            continue

    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    frozen = sum(
        parameter.numel()
        for parameter in model.parameters()
        if not parameter.requires_grad
    )

    print(
        f"Damage stage trainable parameters: "
        f"{trainable:,}"
    )

    print(
        f"Damage stage frozen parameters: "
        f"{frozen:,}"
    )

    return model


# ============================================================
# CONFUSION MATRICES
# ============================================================

def update_confusion_matrix(
    confusion_matrix,
    prediction,
    target,
    valid_mask,
    num_classes,
):

    prediction = prediction.reshape(-1)
    target = target.reshape(-1)
    valid_mask = valid_mask.reshape(-1)

    selected = (
        valid_mask.bool()
        &
        (target >= 0)
        &
        (target < num_classes)
        &
        (prediction >= 0)
        &
        (prediction < num_classes)
    )

    prediction = prediction[
        selected
    ].long()

    target = target[
        selected
    ].long()

    if target.numel() == 0:
        return confusion_matrix

    batch_matrix = torch.bincount(
        num_classes * target +
        prediction,
        minlength=num_classes**2,
    ).reshape(
        num_classes,
        num_classes,
    )

    confusion_matrix += (
        batch_matrix.detach().cpu()
    )

    return confusion_matrix


def update_event_confusion_matrices(
    event_matrices,
    prediction,
    target,
    valid_mask,
    event_names,
):

    unique_events = sorted(
        set(event_names)
    )

    for event in unique_events:

        sample_indices = [
            index
            for index, name in enumerate(
                event_names
            )
            if name == event
        ]

        index_tensor = torch.tensor(
            sample_indices,
            device=prediction.device,
            dtype=torch.long,
        )

        if event not in event_matrices:
            event_matrices[event] = (
                torch.zeros(
                    NUM_DAMAGE_CLASSES,
                    NUM_DAMAGE_CLASSES,
                    dtype=torch.long,
                )
            )

        event_matrices[event] = (
            update_confusion_matrix(
                event_matrices[event],
                prediction.index_select(
                    0,
                    index_tensor,
                ),
                target.index_select(
                    0,
                    index_tensor,
                ),
                valid_mask.index_select(
                    0,
                    index_tensor,
                ),
                NUM_DAMAGE_CLASSES,
            )
        )

    return event_matrices


# ============================================================
# DAMAGE METRICS
# ============================================================

def compute_damage_metrics(
    confusion_matrix,
):

    cm = confusion_matrix.float()

    class_ids = torch.tensor(
        [1, 2, 3],
        dtype=torch.long,
    )

    true_positive = cm.diag()[
        class_ids
    ]

    false_positive = (
        cm[:, class_ids].sum(dim=0)
        -
        true_positive
    )

    # Includes predictions of background on true buildings.
    false_negative = (
        cm[class_ids, :].sum(dim=1)
        -
        true_positive
    )

    precision = (
        true_positive /
        (
            true_positive +
            false_positive +
            1e-6
        )
    )

    recall = (
        true_positive /
        (
            true_positive +
            false_negative +
            1e-6
        )
    )

    f1 = (
        2.0 * precision * recall /
        (
            precision +
            recall +
            1e-6
        )
    )

    iou = (
        true_positive /
        (
            true_positive +
            false_positive +
            false_negative +
            1e-6
        )
    )

    total = cm.sum()

    accuracy = (
        true_positive.sum() /
        (total + 1e-6)
    )

    observed_agreement = (
        cm.diag().sum() /
        (total + 1e-6)
    )

    expected_agreement = (
        cm.sum(dim=0) *
        cm.sum(dim=1)
    ).sum() / (
        total**2 + 1e-6
    )

    kappa = (
        observed_agreement -
        expected_agreement
    ) / (
        1.0 -
        expected_agreement +
        1e-6
    )

    return {
        "accuracy": accuracy.item(),
        "precision_macro": (
            precision.mean().item()
        ),
        "recall_macro": (
            recall.mean().item()
        ),
        "f1_macro": f1.mean().item(),
        "kappa": kappa.item(),
        "miou_3class": (
            iou.mean().item()
        ),
        "miou_rare_2class": (
            (
                iou[1] +
                iou[2]
            ) / 2.0
        ).item(),
        "precision_per_class": (
            precision.numpy()
        ),
        "recall_per_class": (
            recall.numpy()
        ),
        "f1_per_class": f1.numpy(),
        "iou_per_class": iou.numpy(),
        "evaluated_pixels": int(
            total.item()
        ),
    }


# ============================================================
# LOCALIZATION METRIC
# ============================================================

def building_iou(
    logits,
    building_target,
):

    prediction = torch.argmax(
        logits,
        dim=1,
    )

    predicted_building = (
        prediction == 1
    )

    reference_building = (
        building_target == 1
    )

    intersection = (
        predicted_building &
        reference_building
    ).sum().item()

    union = (
        predicted_building |
        reference_building
    ).sum().item()

    return intersection, union


# ============================================================
# ONE TRAINING EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    criterion,
    loss_weights,
    stage,
):

    model.train()

    optimizer.zero_grad(
        set_to_none=True
    )

    running = defaultdict(float)

    damage_cm = torch.zeros(
        NUM_DAMAGE_CLASSES,
        NUM_DAMAGE_CLASSES,
        dtype=torch.long,
    )

    total_batches = len(loader)

    progress_bar = tqdm(
        enumerate(loader),
        total=total_batches,
        desc=f"Training {stage}",
        leave=False,
    )

    for batch_index, batch in progress_bar:

        (
            pre_image,
            post_image,
            damage_target,
            valid_mask,
            event_names,
        ) = batch

        pre_image = pre_image.to(
            DEVICE,
            non_blocking=True,
        )

        post_image = post_image.to(
            DEVICE,
            non_blocking=True,
        )

        damage_target = damage_target.to(
            DEVICE,
            non_blocking=True,
        )

        valid_mask = valid_mask.to(
            DEVICE,
            non_blocking=True,
        )

        building_target = (
            valid_mask.long()
        )

        with autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):

            (
                pre_building_logits,
                post_building_logits,
                raw_damage_logits,
            ) = model(
                pre_image,
                post_image,
            )

            damage_logits = (
                apply_pre_building_gate(
                    raw_damage_logits,
                    pre_building_logits,
                )
            )

            losses = criterion(
                pre_building_logits=(
                    pre_building_logits
                ),
                post_building_logits=(
                    post_building_logits
                ),
                damage_logits=damage_logits,
                building_target=(
                    building_target
                ),
                damage_target=(
                    damage_target
                ),
                loss_weights=loss_weights,
            )

            total_loss = losses["total"]

        if not torch.isfinite(
            total_loss
        ):
            optimizer.zero_grad(
                set_to_none=True
            )
            print("Skipped non-finite loss.")
            continue

        scaler.scale(
            total_loss /
            GRADIENT_ACCUMULATION_STEPS
        ).backward()

        should_update = (
            (batch_index + 1) %
            GRADIENT_ACCUMULATION_STEPS == 0
            or
            (batch_index + 1) ==
            total_batches
        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [
                    parameter
                    for parameter
                    in model.parameters()
                    if parameter.requires_grad
                ],
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        batch_size = pre_image.size(0)

        for loss_name in [
            "total",
            "pre_building",
            "post_building",
            "damage",
        ]:
            running[loss_name] += (
                losses[loss_name].item() *
                batch_size
            )

        running["samples"] += batch_size

        with torch.no_grad():

            damage_prediction = (
                torch.argmax(
                    damage_logits,
                    dim=1,
                )
            )

            damage_cm = (
                update_confusion_matrix(
                    damage_cm,
                    damage_prediction,
                    damage_target,
                    valid_mask,
                    NUM_DAMAGE_CLASSES,
                )
            )

        progress_bar.set_postfix({
            "loss": (
                f"{total_loss.item():.4f}"
            )
        })

    denominator = max(
        running["samples"],
        1,
    )

    average_losses = {
        name: running[name] /
        denominator
        for name in [
            "total",
            "pre_building",
            "post_building",
            "damage",
        ]
    }

    damage_metrics = (
        compute_damage_metrics(
            damage_cm
        )
    )

    return (
        average_losses,
        damage_metrics,
    )


# ============================================================
# VALIDATION: GLOBAL + EVENT
# ============================================================

@torch.no_grad()
def validate_model(
    model,
    loader,
    criterion,
    loss_weights,
    description="Validation",
):

    model.eval()

    running = defaultdict(float)

    global_cm = torch.zeros(
        NUM_DAMAGE_CLASSES,
        NUM_DAMAGE_CLASSES,
        dtype=torch.long,
    )

    event_cms = {}

    pre_intersection = 0
    pre_union = 0

    post_intersection = 0
    post_union = 0

    for batch in tqdm(
        loader,
        desc=description,
        leave=False,
    ):

        (
            pre_image,
            post_image,
            damage_target,
            valid_mask,
            event_names,
        ) = batch

        pre_image = pre_image.to(
            DEVICE,
            non_blocking=True,
        )

        post_image = post_image.to(
            DEVICE,
            non_blocking=True,
        )

        damage_target = damage_target.to(
            DEVICE,
            non_blocking=True,
        )

        valid_mask = valid_mask.to(
            DEVICE,
            non_blocking=True,
        )

        building_target = (
            valid_mask.long()
        )

        with autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):

            (
                pre_building_logits,
                post_building_logits,
                raw_damage_logits,
            ) = model(
                pre_image,
                post_image,
            )

            damage_logits = (
                apply_pre_building_gate(
                    raw_damage_logits,
                    pre_building_logits,
                )
            )

            losses = criterion(
                pre_building_logits,
                post_building_logits,
                damage_logits,
                building_target,
                damage_target,
                loss_weights,
            )

        batch_size = pre_image.size(0)

        for loss_name in [
            "total",
            "pre_building",
            "post_building",
            "damage",
        ]:
            running[loss_name] += (
                losses[loss_name].item() *
                batch_size
            )

        running["samples"] += batch_size

        pre_i, pre_u = building_iou(
            pre_building_logits,
            building_target,
        )

        post_i, post_u = building_iou(
            post_building_logits,
            building_target,
        )

        pre_intersection += pre_i
        pre_union += pre_u

        post_intersection += post_i
        post_union += post_u

        damage_prediction = torch.argmax(
            damage_logits,
            dim=1,
        )

        global_cm = (
            update_confusion_matrix(
                global_cm,
                damage_prediction,
                damage_target,
                valid_mask,
                NUM_DAMAGE_CLASSES,
            )
        )

        event_cms = (
            update_event_confusion_matrices(
                event_matrices=event_cms,
                prediction=damage_prediction,
                target=damage_target,
                valid_mask=valid_mask,
                event_names=list(
                    event_names
                ),
            )
        )

    denominator = max(
        running["samples"],
        1,
    )

    average_losses = {
        name: running[name] /
        denominator
        for name in [
            "total",
            "pre_building",
            "post_building",
            "damage",
        ]
    }

    localization_metrics = {
        "pre_building_iou": (
            pre_intersection /
            (pre_union + 1e-6)
        ),
        "post_building_iou": (
            post_intersection /
            (post_union + 1e-6)
        ),
    }

    localization_metrics[
        "mean_building_iou"
    ] = (
        localization_metrics[
            "pre_building_iou"
        ]
        +
        localization_metrics[
            "post_building_iou"
        ]
    ) / 2.0

    global_metrics = (
        compute_damage_metrics(
            global_cm
        )
    )

    event_metrics = {
        event: compute_damage_metrics(
            matrix
        )
        for event, matrix
        in sorted(event_cms.items())
    }

    return {
        "losses": average_losses,
        "localization": (
            localization_metrics
        ),
        "global_metrics": (
            global_metrics
        ),
        "event_metrics": (
            event_metrics
        ),
        "global_confusion_matrix": (
            global_cm
        ),
        "event_confusion_matrices": (
            event_cms
        ),
    }


# ============================================================
# PRINT RESULTS
# ============================================================

def print_validation_results(
    stage,
    results,
):

    losses = results["losses"]
    global_metrics = results[
        "global_metrics"
    ]

    print(f"\n===== {stage} =====")

    print(
        f"Total loss: "
        f"{losses['total']:.4f}"
    )

    print(
        f"Pre-building IoU: "
        f"{results['localization']['pre_building_iou']:.4f}"
    )

    print(
        f"Post-building IoU: "
        f"{results['localization']['post_building_iou']:.4f}"
    )

    print(
        f"Macro F1: "
        f"{global_metrics['f1_macro']:.4f}"
    )

    print(
        f"Kappa: "
        f"{global_metrics['kappa']:.4f}"
    )

    print(
        f"mIoU-3class: "
        f"{global_metrics['miou_3class']:.4f}"
    )

    print(
        f"mIoU-rare-2class: "
        f"{global_metrics['miou_rare_2class']:.4f}"
    )

    for index, class_name in enumerate([
        "Intact",
        "Damage",
        "Destroyed",
    ]):
        print(
            f"{class_name}: "
            f"F1={global_metrics['f1_per_class'][index]:.4f}, "
            f"IoU={global_metrics['iou_per_class'][index]:.4f}"
        )


# ============================================================
# EXCEL: GLOBAL ROW FOLLOWED BY EVENT ROWS
# ============================================================

def metrics_to_row(
    evaluation_scope,
    event_name,
    metrics,
    best_epoch,
):

    row = {
        "evaluation_scope": (
            evaluation_scope
        ),
        "event": event_name,
        "model": "Siamese_UNet",
        "input": "Pre_Post",
        "target_budget_per_class": 500,
        "best_epoch": best_epoch,
        "evaluated_pixels": (
            metrics["evaluated_pixels"]
        ),
        "accuracy": metrics["accuracy"],
        "precision_macro": (
            metrics["precision_macro"]
        ),
        "recall_macro": (
            metrics["recall_macro"]
        ),
        "f1_macro": (
            metrics["f1_macro"]
        ),
        "kappa": metrics["kappa"],
        "miou_3class": (
            metrics["miou_3class"]
        ),
        "miou_rare_2class": (
            metrics[
                "miou_rare_2class"
            ]
        ),
    }

    class_names = [
        "intact",
        "damage",
        "destroyed",
    ]

    for index, class_name in enumerate(
        class_names
    ):

        row[
            f"precision_{class_name}"
        ] = metrics[
            "precision_per_class"
        ][index]

        row[
            f"recall_{class_name}"
        ] = metrics[
            "recall_per_class"
        ][index]

        row[
            f"f1_{class_name}"
        ] = metrics[
            "f1_per_class"
        ][index]

        row[
            f"iou_{class_name}"
        ] = metrics[
            "iou_per_class"
        ][index]

    return row


def save_global_and_event_excel(
    validation_results,
    best_epoch,
):

    rows = []

    # First row: global aggregation
    rows.append(
        metrics_to_row(
            evaluation_scope=(
                "Global"
            ),
            event_name=(
                "Global_All_Events"
            ),
            metrics=validation_results[
                "global_metrics"
            ],
            best_epoch=best_epoch,
        )
    )

    # Following rows: individual events
    for event_name, metrics in (
        validation_results[
            "event_metrics"
        ].items()
    ):

        rows.append(
            metrics_to_row(
                evaluation_scope=(
                    "Event"
                ),
                event_name=event_name,
                metrics=metrics,
                best_epoch=best_epoch,
            )
        )

    metrics_dataframe = (
        pd.DataFrame(rows)
    )

    global_cm_dataframe = (
        pd.DataFrame(
            validation_results[
                "global_confusion_matrix"
            ].numpy(),
            index=[
                "True_BG",
                "True_Intact",
                "True_Damage",
                "True_Destroyed",
            ],
            columns=[
                "Pred_BG",
                "Pred_Intact",
                "Pred_Damage",
                "Pred_Destroyed",
            ],
        )
    )

    with pd.ExcelWriter(
        FINAL_EXCEL_PATH,
        engine="openpyxl",
    ) as writer:

        metrics_dataframe.to_excel(
            writer,
            sheet_name="Metrics",
            index=False,
        )

        global_cm_dataframe.to_excel(
            writer,
            sheet_name=(
                "Global_Confusion_Matrix"
            ),
        )

        for event_name, matrix in (
            validation_results[
                "event_confusion_matrices"
            ].items()
        ):

            event_cm_dataframe = (
                pd.DataFrame(
                    matrix.numpy(),
                    index=[
                        "True_BG",
                        "True_Intact",
                        "True_Damage",
                        "True_Destroyed",
                    ],
                    columns=[
                        "Pred_BG",
                        "Pred_Intact",
                        "Pred_Damage",
                        "Pred_Destroyed",
                    ],
                )
            )

            safe_sheet_name = (
                event_name[:31]
            )

            event_cm_dataframe.to_excel(
                writer,
                sheet_name=safe_sheet_name,
            )

    print(
        f"Final Excel saved: "
        f"{FINAL_EXCEL_PATH}"
    )

    return metrics_dataframe


# ============================================================
# GENERIC STAGE TRAINER
# ============================================================

def train_stage(
    model,
    train_loader,
    val_loader,
    criterion,
    stage_name,
    epochs,
    learning_rate,
    loss_weights,
    selection_metric,
    best_checkpoint_path,
):

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.Adam(
        trainable_parameters,
        lr=learning_rate,
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            patience=3,
            factor=0.5,
            min_lr=MIN_LEARNING_RATE,
        )
    )

    scaler = GradScaler(
        device=DEVICE.type,
        enabled=USE_AMP,
    )

    best_score = -float("inf")
    best_epoch = -1
    no_improvement = 0

    history = {
        "stage": stage_name,
        "train_losses": [],
        "val_epochs": [],
        "val_losses": [],
        "val_results": [],
        "learning_rates": [],
    }

    for epoch in range(epochs):

        current_lr = (
            optimizer.param_groups[0][
                "lr"
            ]
        )

        print(
            f"\n{stage_name} — "
            f"Epoch {epoch + 1}/{epochs} "
            f"| LR={current_lr:.8f}"
        )

        train_losses, train_metrics = (
            train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scaler=scaler,
                criterion=criterion,
                loss_weights=(
                    loss_weights
                ),
                stage=stage_name,
            )
        )

        history[
            "train_losses"
        ].append(train_losses)

        history[
            "learning_rates"
        ].append(current_lr)

        print(
            f"Train total loss: "
            f"{train_losses['total']:.4f}"
        )

        should_validate = (
            (epoch + 1) % EVAL_EVERY == 0
            or
            (epoch + 1) == epochs
        )

        if not should_validate:
            continue

        validation_results = (
            validate_model(
                model=model,
                loader=val_loader,
                criterion=criterion,
                loss_weights=(
                    loss_weights
                ),
                description=(
                    f"{stage_name} validation"
                ),
            )
        )

        print_validation_results(
            stage_name,
            validation_results,
        )

        val_loss = validation_results[
            "losses"
        ]["total"]

        scheduler.step(val_loss)

        history[
            "val_epochs"
        ].append(epoch + 1)

        history[
            "val_losses"
        ].append(
            validation_results[
                "losses"
            ]
        )

        history[
            "val_results"
        ].append(
            validation_results[
                "global_metrics"
            ]
        )

        if selection_metric == (
            "mean_building_iou"
        ):

            score = validation_results[
                "localization"
            ]["mean_building_iou"]

        elif selection_metric == (
            "rare_miou"
        ):

            score = validation_results[
                "global_metrics"
            ]["miou_rare_2class"]

        else:
            raise ValueError(
                f"Unknown selection metric: "
                f"{selection_metric}"
            )

        if score > best_score:

            best_score = score
            best_epoch = epoch + 1
            no_improvement = 0

            torch.save(
                {
                    "stage": stage_name,
                    "epoch": epoch,
                    "best_epoch": best_epoch,
                    "best_score": best_score,
                    "model": (
                        model.state_dict()
                    ),
                    "optimizer": (
                        optimizer.state_dict()
                    ),
                    "scheduler": (
                        scheduler.state_dict()
                    ),
                    "scaler": (
                        scaler.state_dict()
                    ),
                    "loss_weights": (
                        loss_weights
                    ),
                    "validation_results": (
                        validation_results
                    ),
                    "history": history,
                },
                best_checkpoint_path,
            )

            print(
                f"Saved best {stage_name} "
                f"checkpoint: "
                f"score={best_score:.4f}"
            )

        else:

            no_improvement += 1

            print(
                f"No improvement: "
                f"{no_improvement}/"
                f"{EARLY_STOP_PATIENCE}"
            )

        if (
            no_improvement >=
            EARLY_STOP_PATIENCE
        ):
            print(
                f"Early stopping during "
                f"{stage_name} at epoch "
                f"{epoch + 1}."
            )
            break

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not Path(
        best_checkpoint_path
    ).exists():
        raise RuntimeError(
            f"No checkpoint saved for "
            f"{stage_name}."
        )

    best_checkpoint = torch.load(
        best_checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model"]
    )

    return (
        model,
        history,
        best_checkpoint["best_epoch"],
    )


# ============================================================
# COMPLETE TWO-STAGE PROCEDURE
# ============================================================

def train_microsoft_siamese_unet():
    """
    Define and execute the complete two-stage Microsoft
    Siamese U-Net experiment.

    This function is intentionally NOT called in this cell.
    """

    (
        train_loader,
        val_loader,
        train_files,
        val_files,
    ) = build_dataloaders()

    model = SiameseUNet(
        in_channels=3,
        building_classes=(
            NUM_BUILDING_CLASSES
        ),
        damage_classes=(
            NUM_DAMAGE_CLASSES
        ),
        init_features=(
            INITIAL_FEATURES
        ),
    ).to(DEVICE)

    criterion = MicrosoftSiameseLoss(
        building_class_weights=(
            BUILDING_CLASS_WEIGHTS
        ),
        damage_class_weights=(
            DAMAGE_CLASS_WEIGHTS
        ),
    ).to(DEVICE)

    # --------------------------------------------------------
    # Stage 1: building localization
    # --------------------------------------------------------

    print(
        "\n========== STAGE 1: "
        "LOCALIZATION =========="
    )

    (
        model,
        localization_history,
        best_localization_epoch,
    ) = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        stage_name="localization",
        epochs=LOCALIZATION_EPOCHS,
        learning_rate=(
            LOCALIZATION_LEARNING_RATE
        ),
        loss_weights=(
            LOCALIZATION_LOSS_WEIGHTS
        ),
        selection_metric=(
            "mean_building_iou"
        ),
        best_checkpoint_path=(
            BEST_LOCALIZATION_PATH
        ),
    )

    # --------------------------------------------------------
    # Stage 2: freeze localization and train damage decoder
    # --------------------------------------------------------

    print(
        "\n========== STAGE 2: "
        "DAMAGE CLASSIFICATION =========="
    )

    model = (
        freeze_localization_components(
            model
        )
    )

    (
        model,
        damage_history,
        best_damage_epoch,
    ) = train_stage(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        stage_name="damage",
        epochs=DAMAGE_EPOCHS,
        learning_rate=(
            DAMAGE_LEARNING_RATE
        ),
        loss_weights=(
            DAMAGE_LOSS_WEIGHTS
        ),
        selection_metric="rare_miou",
        best_checkpoint_path=(
            BEST_DAMAGE_PATH
        ),
    )

    # --------------------------------------------------------
    # Final global and event-level evaluation
    # --------------------------------------------------------

    final_results = validate_model(
        model=model,
        loader=val_loader,
        criterion=criterion,
        loss_weights=(
            DAMAGE_LOSS_WEIGHTS
        ),
        description=(
            "Final global and event validation"
        ),
    )

    print_validation_results(
        "FINAL DAMAGE EVALUATION",
        final_results,
    )

    metrics_dataframe = (
        save_global_and_event_excel(
            validation_results=(
                final_results
            ),
            best_epoch=(
                best_damage_epoch
            ),
        )
    )

    torch.save(
        {
            "localization_history": (
                localization_history
            ),
            "damage_history": (
                damage_history
            ),
            "best_localization_epoch": (
                best_localization_epoch
            ),
            "best_damage_epoch": (
                best_damage_epoch
            ),
            "final_results": (
                final_results
            ),
            "train_files": [
                os.path.basename(path)
                for path in train_files
            ],
            "val_files": [
                os.path.basename(path)
                for path in val_files
            ],
        },
        HISTORY_PATH,
    )

    return {
        "model": model,
        "localization_history": (
            localization_history
        ),
        "damage_history": (
            damage_history
        ),
        "best_localization_epoch": (
            best_localization_epoch
        ),
        "best_damage_epoch": (
            best_damage_epoch
        ),
        "final_results": final_results,
        "metrics_dataframe": (
            metrics_dataframe
        ),
    }

## Main and Training Step

In [15]:
# ============================================================
# PRE/POST BASELINE WORKSPACE
# ============================================================

from pathlib import Path
from datetime import datetime


# Main workspace containing all bi-temporal baselines
PREPOST_WORKSPACE_ROOT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines"
)


# Four selected models
PREPOST_MODEL_NAMES = [
    "Siamese_UNet",
    "ChangeOS_R18",
    "ChangeFormerV6",
    "MambaBDA_Tiny",
]


# Current experiment
CURRENT_MODEL_NAME = "Siamese_UNet"

TARGET_LABELS_PER_CLASS = 500
SEED = 42

CURRENT_RUN_NAME = (
    f"budget{TARGET_LABELS_PER_CLASS}_"
    f"seed{SEED}"
)


# Create top-level folders for all four models
PREPOST_WORKSPACE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for model_name in PREPOST_MODEL_NAMES:

    model_root = (
        PREPOST_WORKSPACE_ROOT /
        model_name
    )

    model_root.mkdir(
        parents=True,
        exist_ok=True,
    )


print(
    "Pre/Post workspace:",
    PREPOST_WORKSPACE_ROOT,
)

print("\nModel folders:")

for model_name in PREPOST_MODEL_NAMES:
    print(
        " ",
        PREPOST_WORKSPACE_ROOT /
        model_name
    )


# ============================================================
# GENERIC PRE/POST EXPERIMENT MANAGER
# ============================================================

import os
import sys
import json

import numpy as np

from pathlib import Path
from datetime import datetime
from scipy import io


# ============================================================
# JSON CONVERSION
# ============================================================

def make_json_serializable(value):

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(value, dict):

        return {
            str(key): make_json_serializable(
                item
            )
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):

        return [
            make_json_serializable(item)
            for item in value
        ]

    return value


# ============================================================
# PREPARE MODEL-SPECIFIC WORKSPACE
# ============================================================

def configure_model_workspace(
    model_name,
    run_name=None,
):
    """
    Create a separate workspace for one pre/post model.

    The global output variables are updated because the
    previously defined training functions use these paths.
    """

    global CURRENT_MODEL_NAME
    global CURRENT_RUN_NAME

    global OUTPUT_DIR

    global MODEL_DIR
    global LOCALIZATION_MODEL_DIR
    global DAMAGE_MODEL_DIR

    global RESULTS_DIR
    global HISTORY_DIR

    global BEST_LOCALIZATION_PATH
    global BEST_DAMAGE_PATH
    global LATEST_LOCALIZATION_PATH
    global LATEST_DAMAGE_PATH

    global FINAL_EXCEL_PATH
    global FINAL_TEXT_PATH
    global FINAL_MAT_PATH
    global FINAL_JSON_PATH
    global HISTORY_PATH

    if model_name not in (
        PREPOST_MODEL_NAMES
    ):
        raise ValueError(
            f"Unknown model: {model_name}. "
            f"Available models: "
            f"{PREPOST_MODEL_NAMES}"
        )

    CURRENT_MODEL_NAME = model_name

    if run_name is None:

        run_name = (
            f"budget"
            f"{TARGET_LABELS_PER_CLASS}_"
            f"seed{SEED}"
        )

    CURRENT_RUN_NAME = run_name

    # --------------------------------------------------------
    # Main run directory
    # --------------------------------------------------------

    OUTPUT_DIR = (
        PREPOST_WORKSPACE_ROOT /
        model_name /
        run_name
    )

    # --------------------------------------------------------
    # Models
    # --------------------------------------------------------

    MODEL_DIR = (
        OUTPUT_DIR /
        "models"
    )

    LOCALIZATION_MODEL_DIR = (
        MODEL_DIR /
        "localization"
    )

    DAMAGE_MODEL_DIR = (
        MODEL_DIR /
        "damage"
    )

    BEST_LOCALIZATION_PATH = (
        LOCALIZATION_MODEL_DIR /
        "best_localization_model.pth"
    )

    BEST_DAMAGE_PATH = (
        DAMAGE_MODEL_DIR /
        "best_damage_model.pth"
    )

    LATEST_LOCALIZATION_PATH = (
        LOCALIZATION_MODEL_DIR /
        "latest_localization_model.pth"
    )

    LATEST_DAMAGE_PATH = (
        DAMAGE_MODEL_DIR /
        "latest_damage_model.pth"
    )

    # --------------------------------------------------------
    # Results and history
    # --------------------------------------------------------

    RESULTS_DIR = (
        OUTPUT_DIR /
        "results"
    )

    HISTORY_DIR = (
        OUTPUT_DIR /
        "history"
    )

    FINAL_EXCEL_PATH = (
        RESULTS_DIR /
        "final_validation_global_and_event_metrics.xlsx"
    )

    FINAL_TEXT_PATH = (
        RESULTS_DIR /
        "final_results.txt"
    )

    FINAL_MAT_PATH = (
        RESULTS_DIR /
        "final_results.mat"
    )

    FINAL_JSON_PATH = (
        RESULTS_DIR /
        "final_results.json"
    )

    HISTORY_PATH = (
        HISTORY_DIR /
        "training_history.pth"
    )

    # --------------------------------------------------------
    # Create directories
    # --------------------------------------------------------

    for directory in [
        OUTPUT_DIR,
        MODEL_DIR,
        LOCALIZATION_MODEL_DIR,
        DAMAGE_MODEL_DIR,
        RESULTS_DIR,
        HISTORY_DIR,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    workspace = {
        "model_name": model_name,
        "run_name": run_name,
        "output_dir": OUTPUT_DIR,
        "model_dir": MODEL_DIR,
        "localization_model_dir": (
            LOCALIZATION_MODEL_DIR
        ),
        "damage_model_dir": (
            DAMAGE_MODEL_DIR
        ),
        "results_dir": RESULTS_DIR,
        "history_dir": HISTORY_DIR,
    }

    print("=" * 72)
    print("PRE/POST MODEL WORKSPACE")
    print("=" * 72)

    print(
        f"Model:       {model_name}"
    )

    print(
        f"Run:         {run_name}"
    )

    print(
        f"Budget:      "
        f"{TARGET_LABELS_PER_CLASS} "
        f"samples per class"
    )

    print(
        f"Seed:        {SEED}"
    )

    print(
        f"Output:      {OUTPUT_DIR}"
    )

    print(
        f"Models:      {MODEL_DIR}"
    )

    print(
        f"Results:     {RESULTS_DIR}"
    )

    print(
        f"History:     {HISTORY_DIR}"
    )

    print("=" * 72)

    return workspace


# ============================================================
# SAVE TEXT RESULTS
# ============================================================

def save_results_text(
    experiment_results,
    elapsed_seconds,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    global_metrics = (
        final_results[
            "global_metrics"
        ]
    )

    event_metrics = (
        final_results[
            "event_metrics"
        ]
    )

    with open(
        FINAL_TEXT_PATH,
        mode="w",
        encoding="utf-8",
    ) as text_file:

        text_file.write(
            "Pre/Post Building Damage "
            "Assessment Results\n"
        )

        text_file.write(
            "=" * 60 + "\n"
        )

        text_file.write(
            f"Model: "
            f"{CURRENT_MODEL_NAME}\n"
        )

        text_file.write(
            f"Run: "
            f"{CURRENT_RUN_NAME}\n"
        )

        text_file.write(
            f"Target budget per class: "
            f"{TARGET_LABELS_PER_CLASS}\n"
        )

        text_file.write(
            f"Seed: {SEED}\n"
        )

        text_file.write(
            f"Runtime seconds: "
            f"{elapsed_seconds:.2f}\n"
        )

        text_file.write(
            f"Output directory: "
            f"{OUTPUT_DIR}\n\n"
        )

        text_file.write(
            "GLOBAL RESULTS\n"
        )

        text_file.write(
            "-" * 60 + "\n"
        )

        for metric_name in [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "kappa",
            "miou_3class",
            "miou_rare_2class",
        ]:

            text_file.write(
                f"{metric_name}: "
                f"{global_metrics[metric_name]}\n"
            )

        class_names = [
            "intact",
            "damage",
            "destroyed",
        ]

        text_file.write(
            "\nGLOBAL CLASS-WISE RESULTS\n"
        )

        text_file.write(
            "-" * 60 + "\n"
        )

        for index, class_name in enumerate(
            class_names
        ):

            text_file.write(
                f"{class_name}_precision: "
                f"{global_metrics['precision_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_recall: "
                f"{global_metrics['recall_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_f1: "
                f"{global_metrics['f1_per_class'][index]}\n"
            )

            text_file.write(
                f"{class_name}_iou: "
                f"{global_metrics['iou_per_class'][index]}\n"
            )

        text_file.write(
            "\nEVENT-LEVEL RESULTS\n"
        )

        text_file.write(
            "=" * 60 + "\n"
        )

        for event_name, metrics in (
            event_metrics.items()
        ):

            text_file.write(
                f"\n{event_name}\n"
            )

            text_file.write(
                "-" * 60 + "\n"
            )

            for metric_name in [
                "accuracy",
                "precision_macro",
                "recall_macro",
                "f1_macro",
                "kappa",
                "miou_3class",
                "miou_rare_2class",
            ]:

                text_file.write(
                    f"{metric_name}: "
                    f"{metrics[metric_name]}\n"
                )

    print(
        f"Text results saved: "
        f"{FINAL_TEXT_PATH}"
    )


# ============================================================
# SAVE MATLAB RESULTS
# ============================================================

def save_results_mat(
    experiment_results,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    global_metrics = (
        final_results[
            "global_metrics"
        ]
    )

    mat_results = {
        "model_name": (
            CURRENT_MODEL_NAME
        ),
        "run_name": CURRENT_RUN_NAME,
        "target_budget_per_class": (
            TARGET_LABELS_PER_CLASS
        ),
        "seed": SEED,
        "global_accuracy": (
            global_metrics["accuracy"]
        ),
        "global_precision_macro": (
            global_metrics[
                "precision_macro"
            ]
        ),
        "global_recall_macro": (
            global_metrics[
                "recall_macro"
            ]
        ),
        "global_f1_macro": (
            global_metrics["f1_macro"]
        ),
        "global_kappa": (
            global_metrics["kappa"]
        ),
        "global_miou_3class": (
            global_metrics[
                "miou_3class"
            ]
        ),
        "global_miou_rare_2class": (
            global_metrics[
                "miou_rare_2class"
            ]
        ),
        "global_precision_per_class": (
            global_metrics[
                "precision_per_class"
            ]
        ),
        "global_recall_per_class": (
            global_metrics[
                "recall_per_class"
            ]
        ),
        "global_f1_per_class": (
            global_metrics[
                "f1_per_class"
            ]
        ),
        "global_iou_per_class": (
            global_metrics[
                "iou_per_class"
            ]
        ),
        "global_confusion_matrix": (
            final_results[
                "global_confusion_matrix"
            ].numpy()
        ),
    }

    for event_name, metrics in (
        final_results[
            "event_metrics"
        ].items()
    ):

        safe_event_name = (
            event_name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        mat_results[
            f"{safe_event_name}_f1_macro"
        ] = metrics["f1_macro"]

        mat_results[
            f"{safe_event_name}_kappa"
        ] = metrics["kappa"]

        mat_results[
            f"{safe_event_name}_miou_3class"
        ] = metrics[
            "miou_3class"
        ]

        mat_results[
            f"{safe_event_name}_miou_rare_2class"
        ] = metrics[
            "miou_rare_2class"
        ]

        mat_results[
            f"{safe_event_name}_iou_per_class"
        ] = metrics[
            "iou_per_class"
        ]

    io.savemat(
        FINAL_MAT_PATH,
        mat_results,
    )

    print(
        f"MAT results saved: "
        f"{FINAL_MAT_PATH}"
    )


# ============================================================
# SAVE JSON RESULTS
# ============================================================

def save_results_json(
    experiment_results,
    elapsed_seconds,
):

    final_results = (
        experiment_results[
            "final_results"
        ]
    )

    json_results = {
        "model": CURRENT_MODEL_NAME,
        "run": CURRENT_RUN_NAME,
        "target_budget_per_class": (
            TARGET_LABELS_PER_CLASS
        ),
        "seed": SEED,
        "runtime_seconds": (
            elapsed_seconds
        ),
        "best_localization_epoch": (
            experiment_results.get(
                "best_localization_epoch"
            )
        ),
        "best_damage_epoch": (
            experiment_results.get(
                "best_damage_epoch"
            )
        ),
        "global_metrics": (
            final_results[
                "global_metrics"
            ]
        ),
        "event_metrics": (
            final_results[
                "event_metrics"
            ]
        ),
    }

    json_results = (
        make_json_serializable(
            json_results
        )
    )

    with open(
        FINAL_JSON_PATH,
        mode="w",
        encoding="utf-8",
    ) as json_file:

        json.dump(
            json_results,
            json_file,
            indent=2,
            ensure_ascii=False,
        )

    print(
        f"JSON results saved: "
        f"{FINAL_JSON_PATH}"
    )


# ============================================================
# GENERIC EXPERIMENT WRAPPER
# ============================================================

def run_prepost_experiment(
    model_name,
    trainer_function,
    run_name=None,
):
    """
    Configure the output workspace, call the selected trainer,
    and save TXT, MAT, JSON, Excel, checkpoints, and history.

    The trainer itself remains model-specific.
    """

    workspace = configure_model_workspace(
        model_name=model_name,
        run_name=run_name,
    )

    start_time = datetime.now()

    print(
        f"\nStarting {model_name}..."
    )

    sys.stdout.flush()

    experiment_results = (
        trainer_function()
    )

    end_time = datetime.now()

    elapsed_seconds = (
        end_time -
        start_time
    ).total_seconds()

    if experiment_results is None:
        raise RuntimeError(
            f"{model_name} trainer returned None."
        )

    if "final_results" not in (
        experiment_results
    ):
        raise KeyError(
            "Trainer output must contain "
            "'final_results'."
        )

    # Excel is created inside the model-specific trainer.
    # The generic wrapper creates additional portable results.
    save_results_text(
        experiment_results,
        elapsed_seconds,
    )

    save_results_mat(
        experiment_results
    )

    save_results_json(
        experiment_results,
        elapsed_seconds,
    )

    print("\n" + "=" * 72)
    print("EXPERIMENT FINISHED")
    print("=" * 72)

    print(
        f"Model:    {model_name}"
    )

    print(
        f"Runtime:  "
        f"{elapsed_seconds / 3600:.2f} hours"
    )

    print(
        f"Models:   {MODEL_DIR}"
    )

    print(
        f"Results:  {RESULTS_DIR}"
    )

    print(
        f"History:  {HISTORY_DIR}"
    )

    print("=" * 72)

    return experiment_results

Pre/Post workspace: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines

Model folders:
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeOS_R18
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/ChangeFormerV6
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/MambaBDA_Tiny


In [16]:
from pathlib import Path

# ---------------------------------------------------------
# CONFIGURE CURRENT EXPERIMENT WORKSPACE
# ---------------------------------------------------------

workspace_paths = configure_model_workspace(
    model_name="Siamese_UNet",
    run_name="budget500_seed42",
)

print("\nWorkspace configuration completed.")

required_output_paths = [
    OUTPUT_DIR,
    MODEL_DIR,
    LOCALIZATION_MODEL_DIR,
    DAMAGE_MODEL_DIR,
    RESULTS_DIR,
    HISTORY_DIR,
]

for required_path in required_output_paths:
    Path(required_path).mkdir(
        parents=True,
        exist_ok=True,
    )

print("\nVerified output directories:")
for required_path in required_output_paths:
    print(" ", required_path)

PRE/POST MODEL WORKSPACE
Model:       Siamese_UNet
Run:         budget500_seed42
Budget:      500 samples per class
Seed:        42
Output:      /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42
Models:      /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models
Results:     /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results
History:     /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/history

Workspace configuration completed.

Verified output directories:
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/localization
  /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/damage
  /content/dri

## Run

In [17]:
siamese_results = run_prepost_experiment(
    model_name="Siamese_UNet",
    trainer_function=(
        train_microsoft_siamese_unet
    ),
    run_name="budget500_seed42",
)

PRE/POST MODEL WORKSPACE
Model:       Siamese_UNet
Run:         budget500_seed42
Budget:      500 samples per class
Seed:        42
Output:      /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42
Models:      /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models
Results:     /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results
History:     /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/history

Starting Siamese_UNet...


Inspecting training:   0%|          | 0/1380 [00:00<?, ?it/s]


===== TRAINING =====
Total files: 1380
Events:
  EARTHQUAKE-TURKEY: 253
  HURRICANE-DELTA: 396
  HURRICANE-IDA: 653
  TEXAS-TORNADOES: 78
Files containing classes:
  Intact: 1,190
  Damage: 567
  Destroyed: 529


Inspecting validation:   0%|          | 0/3931 [00:00<?, ?it/s]


===== VALIDATION =====
Total files: 3931
Events:
  EARTHQUAKE-TURKEY: 771
  HURRICANE-DELTA: 784
  HURRICANE-IDA: 2,191
  TEXAS-TORNADOES: 185
Files containing classes:
  Intact: 3,812
  Damage: 363
  Destroyed: 293


...

Train total loss: 0.6227


damage validation:   0%|          | 0/123 [00:00<?, ?it/s]


===== damage =====
Total loss: 0.7501
Pre-building IoU: 0.4566
Post-building IoU: 0.3945
Macro F1: 0.4432
Kappa: 0.1725
mIoU-3class: 0.3552
mIoU-rare-2class: 0.1184
Intact: F1=0.9064, IoU=0.8288
Damage: F1=0.1973, IoU=0.1095
Destroyed: F1=0.2260, IoU=0.1274
No improvement: 5/5
Early stopping during damage at epoch 57.


Final global and event validation:   0%|          | 0/123 [00:00<?, ?it/s]


===== FINAL DAMAGE EVALUATION =====
Total loss: 0.6904
Pre-building IoU: 0.4323
Post-building IoU: 0.3754
Macro F1: 0.4729
Kappa: 0.2099
mIoU-3class: 0.3843
mIoU-rare-2class: 0.1376
Intact: F1=0.9349, IoU=0.8778
Damage: F1=0.2319, IoU=0.1312
Destroyed: F1=0.2518, IoU=0.1440
Final Excel saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results/final_validation_global_and_event_metrics.xlsx
Text results saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results/final_results.txt
MAT results saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results/final_results.mat
JSON results saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/results/final_results.json

EXPERIMENT FINISHED
Model:    Siamese_UNet
Runtime:  0.65 hours
Models:   /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models
Results:  /cont

## Predection on independent test event

In [21]:
# ============================================================
# FIX — FETCH ALL FIGSHARE FILES WITH PAGINATION
# ============================================================

import requests


def get_figshare_files(
    article_id,
    page_size=100,
):
    all_files = []
    page = 1

    while True:
        url = (
            f"https://api.figshare.com/v2/articles/"
            f"{article_id}/files"
        )

        response = requests.get(
            url,
            params={
                "page": page,
                "page_size": page_size,
            },
            timeout=120,
        )

        response.raise_for_status()

        page_files = response.json()

        if not isinstance(page_files, list):
            raise RuntimeError(
                f"Invalid Figshare response on page {page}."
            )

        if len(page_files) == 0:
            break

        all_files.extend(page_files)

        print(
            f"Page {page}: "
            f"{len(page_files)} files received"
        )

        if len(page_files) < page_size:
            break

        page += 1

    if len(all_files) == 0:
        raise RuntimeError(
            "Figshare returned no files."
        )

    # حذف موارد تکراری احتمالی براساس file ID
    unique_files = {}

    for file_info in all_files:
        file_id = file_info.get(
            "id",
            file_info.get("name"),
        )

        unique_files[file_id] = file_info

    all_files = list(unique_files.values())

    print("\nTotal unique Figshare files:", len(all_files))

    return all_files

In [22]:
# ============================================================
# FIND DORIAN AFTER COMPLETE PAGINATION
# ============================================================

figshare_files = get_figshare_files(
    ARTICLE_ID,
    page_size=100,
)

print("\nFiles containing DORIAN:")

dorian_candidates = [
    file_info
    for file_info in figshare_files
    if "DORIAN" in file_info["name"].upper()
]

for file_info in dorian_candidates:
    print(
        " ",
        file_info["name"],
        "|",
        f"{file_info.get('size', 0) / 1024**3:.3f} GB",
    )

if len(dorian_candidates) == 0:
    raise FileNotFoundError(
        "No file containing DORIAN was found after "
        "fetching every Figshare API page."
    )

selected_file = find_event_zip(
    files=figshare_files,
    event_name=EVENT_NAME,
)

remote_filename = selected_file["name"]
download_url = selected_file["download_url"]

ZIP_PATH = DOWNLOAD_DIR / remote_filename

print("\nSelected file:")
print("Filename:", remote_filename)
print("URL     :", download_url)
print(
    "Size    :",
    f"{selected_file.get('size', 0) / 1024**3:.3f} GB",
)

Page 1: 12 files received

Total unique Figshare files: 12

Files containing DORIAN:
  HURRICANE-DORIAN.zip | 0.413 GB

Selected file:
Filename: HURRICANE-DORIAN.zip
URL     : https://ndownloader.figshare.com/files/44708263
Size    : 0.413 GB


In [23]:
ZIP_PATH = download_file(
    download_url=download_url,
    destination_path=ZIP_PATH,
)

zip_report = validate_zip(
    ZIP_PATH
)

print("\n" + "=" * 72)
print("STEP 1 COMPLETED SUCCESSFULLY")
print("=" * 72)
print("ZIP path:", ZIP_PATH)
print("Report  :", zip_report)
print("=" * 72)


Downloading:
https://ndownloader.figshare.com/files/44708263



Download completed:
/content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Downloaded size: 0.413 GB

Checking ZIP integrity:
/content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Archive members : 2163
Pre images      : 540
Post images     : 540
Pre masks       : 540
Post masks      : 540

✅ ZIP structure and paired-file counts are valid.

STEP 1 COMPLETED SUCCESSFULLY
ZIP path: /content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Report  : {'archive_members': 2163, 'pre_images': 540, 'post_images': 540, 'pre_masks': 540, 'post_masks': 540}


In [24]:
# ============================================================
# STEP 2 — EXTRACT AND VERIFY PRE/POST PAIRING
# ============================================================

import os
import zipfile

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

ZIP_PATH = (
    LOCAL_ROOT /
    "downloads" /
    "HURRICANE-DORIAN.zip"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

EXTRACT_MARKER = (
    EXTRACT_DIR /
    ".extraction_completed"
)

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. EXTRACT ZIP
# ------------------------------------------------------------

def extract_event_zip(
    zip_path,
    extract_dir,
):
    zip_path = Path(zip_path)
    extract_dir = Path(extract_dir)

    if not zip_path.is_file():
        raise FileNotFoundError(
            f"ZIP file not found:\n{zip_path}"
        )

    if not zipfile.is_zipfile(zip_path):
        raise zipfile.BadZipFile(
            f"Invalid ZIP file:\n{zip_path}"
        )

    existing_png_files = list(
        extract_dir.rglob("*.png")
    )

    # اگر قبلاً کامل استخراج شده، استخراج تکرار نمی‌شود
    if (
        EXTRACT_MARKER.exists()
        and len(existing_png_files) > 0
    ):
        print("Extraction marker found.")
        print("Existing PNG files:", len(existing_png_files))
        print("Skipping repeated extraction.")

        return extract_dir

    print("=" * 72)
    print("EXTRACTING HURRICANE-DORIAN")
    print("=" * 72)
    print("ZIP        :", zip_path)
    print("Destination:", extract_dir)

    with zipfile.ZipFile(zip_path, "r") as archive:
        members = archive.infolist()

        for member in tqdm(
            members,
            desc="Extracting Dorian ZIP",
            unit="file",
        ):
            archive.extract(
                member,
                path=extract_dir,
            )

    EXTRACT_MARKER.write_text(
        "HURRICANE-DORIAN extraction completed.",
        encoding="utf-8",
    )

    print("\nExtraction completed successfully.")
    print("Destination:", extract_dir)

    return extract_dir


# ------------------------------------------------------------
# 3. CREATE FILE INDEX
# ------------------------------------------------------------

def create_event_file_indexes(
    extract_dir,
):
    extract_dir = Path(extract_dir)

    all_png_files = sorted(
        extract_dir.rglob("*.png")
    )

    image_files = [
        path
        for path in all_png_files
        if path.parent.name.lower() == "images"
    ]

    mask_files = [
        path
        for path in all_png_files
        if path.parent.name.lower() == "masks"
    ]

    pre_images = {
        path.name.replace(
            "_pre_disaster.png",
            "",
        ): path
        for path in image_files
        if path.name.endswith(
            "_pre_disaster.png"
        )
    }

    post_images = {
        path.name.replace(
            "_post_disaster.png",
            "",
        ): path
        for path in image_files
        if path.name.endswith(
            "_post_disaster.png"
        )
    }

    pre_masks = {
        path.name.replace(
            "_pre_disaster.png",
            "",
        ): path
        for path in mask_files
        if path.name.endswith(
            "_pre_disaster.png"
        )
    }

    post_masks = {
        path.name.replace(
            "_post_disaster.png",
            "",
        ): path
        for path in mask_files
        if path.name.endswith(
            "_post_disaster.png"
        )
    }

    indexes = {
        "pre_image": pre_images,
        "post_image": post_images,
        "pre_mask": pre_masks,
        "post_mask": post_masks,
    }

    print("\n" + "=" * 72)
    print("EXTRACTED FILE INVENTORY")
    print("=" * 72)
    print("Total PNG files:", len(all_png_files))
    print("Pre images     :", len(pre_images))
    print("Post images    :", len(post_images))
    print("Pre masks      :", len(pre_masks))
    print("Post masks     :", len(post_masks))
    print("=" * 72)

    return indexes


# ------------------------------------------------------------
# 4. VERIFY COMPLETE PAIRS
# ------------------------------------------------------------

def verify_complete_event_pairs(
    indexes,
):
    all_scene_ids = set()

    for file_index in indexes.values():
        all_scene_ids.update(
            file_index.keys()
        )

    complete_pairs = []
    incomplete_pairs = []

    for scene_id in sorted(all_scene_ids):
        missing_parts = [
            component
            for component, file_index in indexes.items()
            if scene_id not in file_index
        ]

        if len(missing_parts) == 0:
            complete_pairs.append({
                "scene_id":
                    scene_id,

                "pre_image":
                    indexes["pre_image"][scene_id],

                "post_image":
                    indexes["post_image"][scene_id],

                "pre_mask":
                    indexes["pre_mask"][scene_id],

                "post_mask":
                    indexes["post_mask"][scene_id],
            })

        else:
            incomplete_pairs.append({
                "scene_id": scene_id,
                "missing": missing_parts,
            })

    print("\n" + "=" * 72)
    print("PRE/POST PAIRING CHECK")
    print("=" * 72)
    print("Unique scene IDs :", len(all_scene_ids))
    print("Complete pairs   :", len(complete_pairs))
    print("Incomplete pairs :", len(incomplete_pairs))

    if incomplete_pairs:
        print("\nFirst incomplete pairs:")

        for item in incomplete_pairs[:10]:
            print(
                item["scene_id"],
                "missing:",
                item["missing"],
            )

    if complete_pairs:
        example = complete_pairs[0]

        print("\nExample complete pair:")
        print("Scene ID  :", example["scene_id"])
        print("Pre image :", example["pre_image"])
        print("Post image:", example["post_image"])
        print("Pre mask  :", example["pre_mask"])
        print("Post mask :", example["post_mask"])

    print("=" * 72)

    if len(complete_pairs) == 0:
        raise RuntimeError(
            "No complete pre/post pair was found."
        )

    if len(incomplete_pairs) > 0:
        raise RuntimeError(
            "Incomplete pre/post pairs were detected."
        )

    return complete_pairs


# ------------------------------------------------------------
# 5. VERIFY EVENT IDENTIFIERS
# ------------------------------------------------------------

def verify_event_names(
    complete_pairs,
    expected_event,
):
    invalid_scene_ids = [
        pair["scene_id"]
        for pair in complete_pairs
        if not pair["scene_id"].upper().startswith(
            expected_event.upper()
        )
    ]

    print("\nEvent-name validation:")
    print("Expected event    :", expected_event)
    print("Invalid scene IDs :", len(invalid_scene_ids))

    if invalid_scene_ids:
        print("Examples:", invalid_scene_ids[:10])

        raise RuntimeError(
            "Some extracted scenes do not belong to "
            f"{expected_event}."
        )

    print("✅ All scene IDs belong to the requested event.")


# ------------------------------------------------------------
# 6. RUN STEP 2
# ------------------------------------------------------------

extracted_root = extract_event_zip(
    zip_path=ZIP_PATH,
    extract_dir=EXTRACT_DIR,
)

event_indexes = create_event_file_indexes(
    extract_dir=extracted_root,
)

complete_pairs = verify_complete_event_pairs(
    indexes=event_indexes,
)

verify_event_names(
    complete_pairs=complete_pairs,
    expected_event=EVENT_NAME,
)

print("\n" + "=" * 72)
print("STEP 2 COMPLETED SUCCESSFULLY")
print("=" * 72)
print("Extracted root :", extracted_root)
print("Complete pairs:", len(complete_pairs))
print("=" * 72)

EXTRACTING HURRICANE-DORIAN
ZIP        : /content/ebd_HURRICANE-DORIAN_local/downloads/HURRICANE-DORIAN.zip
Destination: /content/ebd_HURRICANE-DORIAN_local/extracted


Extracting Dorian ZIP:   0%|          | 0/2163 [00:00<?, ?file/s]


Extraction completed successfully.
Destination: /content/ebd_HURRICANE-DORIAN_local/extracted

EXTRACTED FILE INVENTORY
Total PNG files: 2160
Pre images     : 540
Post images    : 540
Pre masks      : 540
Post masks     : 540

PRE/POST PAIRING CHECK
Unique scene IDs : 540
Complete pairs   : 540
Incomplete pairs : 0

Example complete pair:
Scene ID  : HURRICANE-DORIAN_000011
Pre image : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/images/HURRICANE-DORIAN_000011_pre_disaster.png
Post image: /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/images/HURRICANE-DORIAN_000011_post_disaster.png
Pre mask  : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/masks/HURRICANE-DORIAN_000011_pre_disaster.png
Post mask : /content/ebd_HURRICANE-DORIAN_local/extracted/HURRICANE-DORIAN/masks/HURRICANE-DORIAN_000011_post_disaster.png

Event-name validation:
Expected event    : HURRICANE-DORIAN
Invalid scene IDs : 0
✅ All scene IDs belong to the requested event.


In [25]:
# ============================================================
# STEP 3 — CREATE MATCHED PRE/POST 512-VALID TEST PATCHES
# ============================================================

import os
import glob
import shutil
import cv2
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter, defaultdict


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"

LOCAL_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local"
)

EXTRACT_DIR = (
    LOCAL_ROOT /
    "extracted"
)

PAIRED_TEST_ROOT = (
    LOCAL_ROOT /
    f"{EVENT_NAME}_prepost_npz_512valid_as_256"
)

MANIFEST_PATH = (
    PAIRED_TEST_ROOT /
    "test_manifest.csv"
)

PATCH_SIZE = 256
BLOCK_SIZE = 512

NUM_CLASSES = 4
REQUIRED_DAMAGE_CLASSES = {1, 2, 3}

# در اولین اجرا False بماند.
# فقط اگر قصد بازسازی کامل پوشه را داری True کن.
RESET_OUTPUT = False


# ------------------------------------------------------------
# 2. SAFE OUTPUT PREPARATION
# ------------------------------------------------------------

if RESET_OUTPUT and PAIRED_TEST_ROOT.is_dir():
    print("Removing previous paired test directory:")
    print(PAIRED_TEST_ROOT)

    shutil.rmtree(PAIRED_TEST_ROOT)

PAIRED_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

existing_npz_files = sorted(
    PAIRED_TEST_ROOT.glob("*.npz")
)

if existing_npz_files and not RESET_OUTPUT:
    raise RuntimeError(
        f"{len(existing_npz_files)} existing NPZ files were found in:\n"
        f"{PAIRED_TEST_ROOT}\n\n"
        "To prevent mixing old and new test data, either use the "
        "existing completed dataset or set RESET_OUTPUT=True."
    )


# ------------------------------------------------------------
# 3. DAMAGE-LABEL CONVERSION
# ------------------------------------------------------------

def convert_damage_to_3class_plus_background(
    original_damage,
):
    """
    Original EBD
    ------------
    0 = background
    1 = no damage
    2 = minor damage
    3 = major damage
    4 = destroyed

    STRATA classes
    --------------
    0 = background
    1 = intact = no damage + minor
    2 = damage = major damage
    3 = destroyed
    """
    converted = np.zeros_like(
        original_damage,
        dtype=np.uint8,
    )

    converted[
        (original_damage == 1)
        | (original_damage == 2)
    ] = 1

    converted[
        original_damage == 3
    ] = 2

    converted[
        original_damage == 4
    ] = 3

    return converted


# ------------------------------------------------------------
# 4. VALID 512-BLOCK CONDITION
# ------------------------------------------------------------

def block_has_required_classes(
    damage_block,
    building_block,
    required_classes,
):
    valid_building = building_block > 0

    if not np.any(valid_building):
        return False

    classes_inside_buildings = set(
        np.unique(
            damage_block[valid_building]
        ).astype(int).tolist()
    )

    return required_classes.issubset(
        classes_inside_buildings
    )


# ------------------------------------------------------------
# 5. READ AND CHECK A COMPLETE SCENE
# ------------------------------------------------------------

def read_complete_scene(pair):
    pre_image = cv2.imread(
        str(pair["pre_image"]),
        cv2.IMREAD_COLOR,
    )

    post_image = cv2.imread(
        str(pair["post_image"]),
        cv2.IMREAD_COLOR,
    )

    pre_mask = cv2.imread(
        str(pair["pre_mask"]),
        cv2.IMREAD_GRAYSCALE,
    )

    post_mask = cv2.imread(
        str(pair["post_mask"]),
        cv2.IMREAD_GRAYSCALE,
    )

    arrays = {
        "pre_image": pre_image,
        "post_image": post_image,
        "pre_mask": pre_mask,
        "post_mask": post_mask,
    }

    missing = [
        name
        for name, array in arrays.items()
        if array is None
    ]

    if missing:
        raise RuntimeError(
            f"Could not read {missing} for scene "
            f"{pair['scene_id']}"
        )

    spatial_shapes = {
        pre_image.shape[:2],
        post_image.shape[:2],
        pre_mask.shape,
        post_mask.shape,
    }

    if len(spatial_shapes) != 1:
        raise RuntimeError(
            f"Spatial-shape mismatch for scene "
            f"{pair['scene_id']}:\n"
            f"pre_image={pre_image.shape}, "
            f"post_image={post_image.shape}, "
            f"pre_mask={pre_mask.shape}, "
            f"post_mask={post_mask.shape}"
        )

    return (
        pre_image,
        post_image,
        pre_mask,
        post_mask,
    )


# ------------------------------------------------------------
# 6. CREATE PAIRED TEST PATCHES
# ------------------------------------------------------------

def create_paired_512valid_test_patches(
    complete_pairs,
):
    if not complete_pairs:
        raise RuntimeError(
            "complete_pairs is empty. Run Step 2 first."
        )

    saved_blocks = 0
    saved_patches = 0

    skipped_missing_file = 0
    skipped_bad_shape = 0
    skipped_incomplete_512 = 0
    skipped_no_building = 0
    skipped_missing_classes = 0
    skipped_black_pre = 0
    skipped_black_post = 0

    original_damage_values = Counter()
    converted_damage_values = Counter()

    manifest_rows = []

    for pair in tqdm(
        complete_pairs,
        desc="Preparing paired Dorian test data",
        unit="scene",
    ):
        scene_id = pair["scene_id"]

        try:
            (
                pre_image,
                post_image,
                pre_mask,
                post_mask,
            ) = read_complete_scene(pair)

        except FileNotFoundError:
            skipped_missing_file += 1
            continue

        except RuntimeError as error:
            print("\nWARNING:", error)
            skipped_bad_shape += 1
            continue

        building_mask = (
            pre_mask > 0
        ).astype(np.uint8)

        damage_mask = (
            convert_damage_to_3class_plus_background(
                post_mask
            )
        )

        for value, count in zip(
            *np.unique(
                post_mask,
                return_counts=True,
            )
        ):
            original_damage_values[int(value)] += int(count)

        for value, count in zip(
            *np.unique(
                damage_mask,
                return_counts=True,
            )
        ):
            converted_damage_values[int(value)] += int(count)

        height, width = damage_mask.shape

        for block_y in range(
            0,
            height,
            BLOCK_SIZE,
        ):
            for block_x in range(
                0,
                width,
                BLOCK_SIZE,
            ):
                pre_block = pre_image[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                post_block = post_image[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                building_block = building_mask[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                damage_block = damage_mask[
                    block_y:block_y + BLOCK_SIZE,
                    block_x:block_x + BLOCK_SIZE,
                ]

                expected_image_shape = (
                    BLOCK_SIZE,
                    BLOCK_SIZE,
                    3,
                )

                expected_mask_shape = (
                    BLOCK_SIZE,
                    BLOCK_SIZE,
                )

                if (
                    pre_block.shape != expected_image_shape
                    or post_block.shape != expected_image_shape
                    or building_block.shape != expected_mask_shape
                    or damage_block.shape != expected_mask_shape
                ):
                    skipped_incomplete_512 += 1
                    continue

                if not np.any(building_block > 0):
                    skipped_no_building += 1
                    continue

                if not block_has_required_classes(
                    damage_block=damage_block,
                    building_block=building_block,
                    required_classes=REQUIRED_DAMAGE_CLASSES,
                ):
                    skipped_missing_classes += 1
                    continue

                # برای گزارش فقط؛ مطابق STRATA این بلوک‌ها حذف نمی‌شوند.
                if np.all(pre_block == 0):
                    skipped_black_pre += 1

                if np.all(post_block == 0):
                    skipped_black_post += 1

                # همان چهار موقعیت کد STRATA
                patch_positions = [
                    (0, 0),
                    (0, PATCH_SIZE),
                    (PATCH_SIZE, 0),
                    (PATCH_SIZE, PATCH_SIZE),
                ]

                block_patch_count = 0

                for local_y, local_x in patch_positions:
                    y = block_y + local_y
                    x = block_x + local_x

                    pre_patch = pre_image[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    post_patch = post_image[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    building_patch = building_mask[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    damage_patch = damage_mask[
                        y:y + PATCH_SIZE,
                        x:x + PATCH_SIZE,
                    ]

                    if (
                        pre_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE, 3)
                        or post_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE, 3)
                        or building_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE)
                        or damage_patch.shape
                        != (PATCH_SIZE, PATCH_SIZE)
                    ):
                        raise RuntimeError(
                            f"Unexpected 256-patch shape: "
                            f"{scene_id}, y={y}, x={x}"
                        )

                    save_name = (
                        f"{scene_id}_{y}_{x}.npz"
                    )

                    save_path = (
                        PAIRED_TEST_ROOT /
                        save_name
                    )

                    np.savez_compressed(
                        save_path,

                        # Siamese U-Net inputs
                        pre_image=pre_patch.astype(
                            np.uint8
                        ),

                        post_image=post_patch.astype(
                            np.uint8
                        ),

                        # Same STRATA quantitative evaluation targets
                        building_mask=building_patch.astype(
                            np.uint8
                        ),

                        damage_mask=damage_patch.astype(
                            np.uint8
                        ),
                    )

                    classes_in_patch = sorted(
                        np.unique(
                            damage_patch[
                                building_patch > 0
                            ]
                        ).astype(int).tolist()
                    ) if np.any(building_patch > 0) else []

                    manifest_rows.append({
                        "event":
                            EVENT_NAME,

                        "scene_id":
                            scene_id,

                        "block_y":
                            block_y,

                        "block_x":
                            block_x,

                        "patch_y":
                            y,

                        "patch_x":
                            x,

                        "filename":
                            save_name,

                        "valid_building_pixels":
                            int(
                                np.sum(
                                    building_patch > 0
                                )
                            ),

                        "classes_inside_building":
                            ",".join(
                                map(str, classes_in_patch)
                            ),

                        "block_contains_all_classes":
                            True,
                    })

                    saved_patches += 1
                    block_patch_count += 1

                if block_patch_count != 4:
                    raise RuntimeError(
                        f"Block {scene_id}, "
                        f"y={block_y}, x={block_x} "
                        f"did not produce exactly four patches."
                    )

                saved_blocks += 1

    manifest_df = pd.DataFrame(
        manifest_rows
    )

    manifest_df.to_csv(
        MANIFEST_PATH,
        index=False,
    )

    print("\n" + "=" * 72)
    print("PAIRED DORIAN TEST PREPARATION SUMMARY")
    print("=" * 72)
    print("Complete scenes examined :", len(complete_pairs))
    print("Valid 512 blocks         :", saved_blocks)
    print("Saved 256 patches        :", saved_patches)
    print("Expected patches         :", saved_blocks * 4)
    print("Skipped missing file     :", skipped_missing_file)
    print("Skipped bad shape        :", skipped_bad_shape)
    print("Skipped incomplete 512   :", skipped_incomplete_512)
    print("Skipped no building      :", skipped_no_building)
    print("Skipped missing classes  :", skipped_missing_classes)
    print("All-black pre blocks     :", skipped_black_pre)
    print("All-black post blocks    :", skipped_black_post)
    print("Output directory         :", PAIRED_TEST_ROOT)
    print("Manifest                 :", MANIFEST_PATH)

    print("\nOriginal post-mask pixel counts:")
    print(dict(sorted(original_damage_values.items())))

    print("\nConverted damage-mask pixel counts:")
    print(dict(sorted(converted_damage_values.items())))

    if saved_blocks == 0:
        raise RuntimeError(
            "No 512 block containing all classes "
            "{1, 2, 3} was found."
        )

    if saved_patches != saved_blocks * 4:
        raise RuntimeError(
            "Saved patch count is inconsistent with "
            "the STRATA 512-valid protocol."
        )

    print("\n✅ Matched paired test data created successfully.")
    print("=" * 72)

    return {
        "root": str(PAIRED_TEST_ROOT),
        "manifest": str(MANIFEST_PATH),
        "valid_512_blocks": saved_blocks,
        "saved_256_patches": saved_patches,
    }


# ------------------------------------------------------------
# 7. RUN STEP 3
# ------------------------------------------------------------

paired_test_report = (
    create_paired_512valid_test_patches(
        complete_pairs=complete_pairs,
    )
)

paired_test_report

Preparing paired Dorian test data:   0%|          | 0/540 [00:00<?, ?scene/s]


PAIRED DORIAN TEST PREPARATION SUMMARY
Complete scenes examined : 540
Valid 512 blocks         : 181
Saved 256 patches        : 724
Expected patches         : 724
Skipped missing file     : 0
Skipped bad shape        : 0
Skipped incomplete 512   : 0
Skipped no building      : 170
Skipped missing classes  : 189
All-black pre blocks     : 0
All-black post blocks    : 0
Output directory         : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256
Manifest                 : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/test_manifest.csv

Original post-mask pixel counts:
{0: 135671818, 1: 564023, 2: 1692972, 3: 2363588, 4: 1265359}

Converted damage-mask pixel counts:
{0: 135671818, 1: 2256995, 2: 2363588, 3: 1265359}

✅ Matched paired test data created successfully.


{'root': '/content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256',
 'manifest': '/content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/test_manifest.csv',
 'valid_512_blocks': 181,
 'saved_256_patches': 724}

In [26]:
# ============================================================
# STEP 3.1 — VALIDATE GENERATED PAIRED NPZ FILES
# ============================================================

import random
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm


def validate_paired_test_npz(
    root_dir,
    full_validation=True,
    random_display_count=5,
    seed=42,
):
    root_dir = Path(root_dir)

    files = sorted(
        root_dir.glob("*.npz")
    )

    if len(files) == 0:
        raise FileNotFoundError(
            f"No paired test NPZ files found in:\n{root_dir}"
        )

    required_keys = {
        "pre_image",
        "post_image",
        "building_mask",
        "damage_mask",
    }

    validation_files = (
        files
        if full_validation
        else files[:min(20, len(files))]
    )

    errors = []

    for path in tqdm(
        validation_files,
        desc="Validating paired test NPZ",
        unit="file",
    ):
        try:
            with np.load(path) as data:
                keys = set(data.keys())

                if keys != required_keys:
                    errors.append(
                        f"{path.name}: keys={sorted(keys)}"
                    )
                    continue

                pre_image = data["pre_image"]
                post_image = data["post_image"]
                building = data["building_mask"]
                damage = data["damage_mask"]

                if pre_image.shape != (256, 256, 3):
                    errors.append(
                        f"{path.name}: invalid pre shape "
                        f"{pre_image.shape}"
                    )

                if post_image.shape != (256, 256, 3):
                    errors.append(
                        f"{path.name}: invalid post shape "
                        f"{post_image.shape}"
                    )

                if building.shape != (256, 256):
                    errors.append(
                        f"{path.name}: invalid building shape "
                        f"{building.shape}"
                    )

                if damage.shape != (256, 256):
                    errors.append(
                        f"{path.name}: invalid damage shape "
                        f"{damage.shape}"
                    )

                if pre_image.dtype != np.uint8:
                    errors.append(
                        f"{path.name}: pre dtype={pre_image.dtype}"
                    )

                if post_image.dtype != np.uint8:
                    errors.append(
                        f"{path.name}: post dtype={post_image.dtype}"
                    )

                if not set(
                    np.unique(building).tolist()
                ).issubset({0, 1}):
                    errors.append(
                        f"{path.name}: invalid building values"
                    )

                if not set(
                    np.unique(damage).tolist()
                ).issubset({0, 1, 2, 3}):
                    errors.append(
                        f"{path.name}: invalid damage values"
                    )

        except Exception as error:
            errors.append(
                f"{path.name}: {repr(error)}"
            )

    print("\n" + "=" * 72)
    print("PAIRED NPZ VALIDATION")
    print("=" * 72)
    print("Files found     :", len(files))
    print("Files validated :", len(validation_files))
    print("Errors          :", len(errors))

    if errors:
        print("\nFirst errors:")

        for error in errors[:20]:
            print(" ", error)

        raise RuntimeError(
            "Paired test NPZ validation failed."
        )

    if len(files) % 4 != 0:
        raise RuntimeError(
            "NPZ count is not divisible by four."
        )

    random_generator = random.Random(seed)

    selected_files = random_generator.sample(
        files,
        k=min(random_display_count, len(files)),
    )

    print("\nRandom valid samples:")

    for path in selected_files:
        with np.load(path) as data:
            print(
                f"  {path.name} | "
                f"damage={np.unique(data['damage_mask']).tolist()} | "
                f"building_pixels="
                f"{int(np.sum(data['building_mask'] > 0))}"
            )

    print("\nComplete 512 groups:", len(files) // 4)
    print("✅ All paired test NPZ files are valid.")
    print("=" * 72)

    return files


paired_test_files = validate_paired_test_npz(
    root_dir=PAIRED_TEST_ROOT,
    full_validation=True,
    random_display_count=5,
    seed=42,
)

Validating paired test NPZ:   0%|          | 0/724 [00:00<?, ?file/s]


PAIRED NPZ VALIDATION
Files found     : 724
Files validated : 724
Errors          : 0

Random valid samples:
  HURRICANE-DORIAN_016657_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=11387
  HURRICANE-DORIAN_003221_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=8442
  HURRICANE-DORIAN_000151_0_256.npz | damage=[0, 1, 2, 3] | building_pixels=2752
  HURRICANE-DORIAN_007846_0_256.npz | damage=[0, 1, 2] | building_pixels=4421
  HURRICANE-DORIAN_006688_256_0.npz | damage=[0, 1, 2, 3] | building_pixels=2606

Complete 512 groups: 181
✅ All paired test NPZ files are valid.


In [27]:
# ============================================================
# STEP 4.1 — TEST SETTINGS AND CHECKPOINT DISCOVERY
# ============================================================

import os
import re
import glob
import torch
import numpy as np

from pathlib import Path


# ------------------------------------------------------------
# 1. TEST SETTINGS
# ------------------------------------------------------------

EVENT_NAME = "HURRICANE-DORIAN"
EVENT_TAG = re.sub(
    r"[^A-Za-z0-9]+",
    "_",
    EVENT_NAME,
).strip("_")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

NUM_CLASSES = 4
CLASS_NAMES = [
    "background",
    "intact",
    "damage",
    "destroyed",
]

FG_CLASS_NAMES = [
    "intact",
    "damage",
    "destroyed",
]

TEST_BATCH_SIZE = 16

# به علت خطای قبلی DataLoader
TEST_NUM_WORKERS = 0

PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# 2. INPUT DATA
# ------------------------------------------------------------

PAIRED_TEST_ROOT = Path(
    f"/content/ebd_{EVENT_NAME}_local/"
    f"{EVENT_NAME}_prepost_npz_512valid_as_256"
)


# ------------------------------------------------------------
# 3. MODEL RUN WORKSPACE
# ------------------------------------------------------------

SIAMESE_RUN_ROOT = Path(
    "/content/drive/MyDrive/STRATA_runs/"
    "PrePost_Baselines/"
    "Siamese_UNet/"
    "budget500_seed42"
)

MODEL_ROOT = (
    SIAMESE_RUN_ROOT /
    "models"
)

TEST_OUTPUT_DIR = (
    SIAMESE_RUN_ROOT /
    f"{EVENT_TAG}_test_metrics"
)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 4. CHECKPOINT DISCOVERY
# ------------------------------------------------------------

def discover_damage_checkpoints(
    model_root,
):
    model_root = Path(model_root)

    if not model_root.is_dir():
        raise FileNotFoundError(
            f"Model directory not found:\n{model_root}"
        )

    checkpoint_extensions = {
        ".pth",
        ".pt",
        ".ckpt",
    }

    all_checkpoints = sorted(
        path
        for path in model_root.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in checkpoint_extensions
        )
    )

    if len(all_checkpoints) == 0:
        raise FileNotFoundError(
            f"No checkpoint was found under:\n{model_root}"
        )

    print("=" * 72)
    print("AVAILABLE SIAMESE U-NET CHECKPOINTS")
    print("=" * 72)

    for index, path in enumerate(
        all_checkpoints,
        start=1,
    ):
        size_mb = path.stat().st_size / 1024**2

        print(
            f"{index:02d}. {path} "
            f"({size_mb:.2f} MB)"
        )

    # اولویت با best checkpoint مربوط به damage
    damage_candidates = [
        path
        for path in all_checkpoints
        if (
            "damage" in str(path).lower()
            and "best" in path.name.lower()
        )
    ]

    if len(damage_candidates) == 0:
        damage_candidates = [
            path
            for path in all_checkpoints
            if "damage" in str(path).lower()
        ]

    if len(damage_candidates) == 0:
        raise FileNotFoundError(
            "No damage-stage checkpoint was found. "
            "Do not use the localization checkpoint for testing."
        )

    # اگر چند فایل بود، جدیدترین best damage انتخاب می‌شود
    damage_candidates = sorted(
        damage_candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    selected_checkpoint = damage_candidates[0]

    print("\nSelected damage checkpoint:")
    print(selected_checkpoint)
    print("=" * 72)

    return selected_checkpoint


BEST_DAMAGE_CHECKPOINT = discover_damage_checkpoints(
    MODEL_ROOT
)

print("\nTest configuration:")
print("Device          :", DEVICE)
print("Test data       :", PAIRED_TEST_ROOT)
print("Checkpoint      :", BEST_DAMAGE_CHECKPOINT)
print("Test output     :", TEST_OUTPUT_DIR)
print("Test batch size :", TEST_BATCH_SIZE)
print("Test workers    :", TEST_NUM_WORKERS)

AVAILABLE SIAMESE U-NET CHECKPOINTS
01. /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/damage/best_damage_model.pth (27.98 MB)
02. /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/localization/best_localization_model.pth (42.85 MB)

Selected damage checkpoint:
/content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/damage/best_damage_model.pth

Test configuration:
Device          : cuda
Test data       : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256
Checkpoint      : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/damage/best_damage_model.pth
Test output     : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics
Test batch size : 16
Test workers    : 0


In [28]:
# ============================================================
# STEP 4.2 — INDEPENDENT PAIRED TEST DATASET
# ============================================================

import os
import glob
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader


# ------------------------------------------------------------
# 1. BASIC UTILITIES
# ------------------------------------------------------------

def normalize_test_image(image):
    """
    Same normalization used during Siamese U-Net training.
    BGR channel order is preserved.
    """
    return image.astype(
        np.float32
    ) / 255.0


def safe_contiguous(array):
    return np.ascontiguousarray(
        array
    )


def get_event_from_filename(path):
    return Path(path).name.split("_")[0]


# ------------------------------------------------------------
# 2. DATASET
# ------------------------------------------------------------

class SiameseIndependentTestDataset(Dataset):

    def __init__(
        self,
        root_dir,
    ):
        self.root_dir = Path(root_dir)

        self.files = sorted(
            self.root_dir.glob("*.npz")
        )

        if len(self.files) == 0:
            raise FileNotFoundError(
                f"No test NPZ files found in:\n"
                f"{self.root_dir}"
            )

    def __len__(self):
        return len(self.files)

    def __getitem__(
        self,
        index,
    ):
        file_path = self.files[index]

        with np.load(file_path) as data:
            pre_image = np.array(
                data["pre_image"],
                dtype=np.uint8,
                copy=True,
            )

            post_image = np.array(
                data["post_image"],
                dtype=np.uint8,
                copy=True,
            )

            damage_mask = np.array(
                data["damage_mask"],
                dtype=np.uint8,
                copy=True,
            )

            building_mask = np.array(
                data["building_mask"],
                dtype=np.uint8,
                copy=True,
            )

        pre_image = normalize_test_image(
            pre_image
        )

        post_image = normalize_test_image(
            post_image
        )

        # HWC -> CHW
        pre_image = np.transpose(
            pre_image,
            (2, 0, 1),
        )

        post_image = np.transpose(
            post_image,
            (2, 0, 1),
        )

        pre_image = safe_contiguous(
            pre_image
        )

        post_image = safe_contiguous(
            post_image
        )

        damage_mask = safe_contiguous(
            damage_mask
        )

        building_mask = safe_contiguous(
            building_mask
        )

        pre_tensor = torch.from_numpy(
            pre_image
        ).float()

        post_tensor = torch.from_numpy(
            post_image
        ).float()

        damage_tensor = torch.from_numpy(
            damage_mask
        ).long()

        building_tensor = torch.from_numpy(
            building_mask
        ).float()

        event_name = get_event_from_filename(
            file_path
        )

        return (
            pre_tensor,
            post_tensor,
            damage_tensor,
            building_tensor,
            event_name,
            str(file_path),
        )


# ------------------------------------------------------------
# 3. DATA LOADER
# ------------------------------------------------------------

test_dataset = SiameseIndependentTestDataset(
    root_dir=PAIRED_TEST_ROOT
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=TEST_NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

print("=" * 72)
print("SIAMESE INDEPENDENT TEST DATA")
print("=" * 72)
print("Event       :", EVENT_NAME)
print("Test patches:", len(test_dataset))
print("Test batches:", len(test_loader))
print("First file  :", test_dataset.files[0])
print("Last file   :", test_dataset.files[-1])
print("=" * 72)

assert len(test_dataset) == 724, (
    f"Expected 724 test patches, "
    f"but found {len(test_dataset)}."
)

SIAMESE INDEPENDENT TEST DATA
Event       : HURRICANE-DORIAN
Test patches: 724
Test batches: 46
First file  : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/HURRICANE-DORIAN_000011_0_0.npz
Last file   : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/HURRICANE-DORIAN_017989_256_256.npz


In [31]:
# ============================================================
# STEP 4.3 — LOAD BEST DAMAGE CHECKPOINT
# ============================================================

import torch
from collections import OrderedDict


def extract_model_state_dict(
    checkpoint,
):
    """
    Supported checkpoint structures:
      checkpoint["model_state_dict"]
      checkpoint["model"]
      checkpoint["state_dict"]
      direct state_dict
    """
    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Checkpoint must be a dictionary."
        )

    candidate_keys = [
        "model_state_dict",
        "model",
        "state_dict",
        "net",
        "network",
    ]

    for key in candidate_keys:
        if (
            key in checkpoint
            and isinstance(checkpoint[key], dict)
        ):
            print(
                f"Model state dictionary found in "
                f"checkpoint['{key}']."
            )

            return checkpoint[key]

    # تشخیص direct state_dict
    if all(
        isinstance(key, str)
        for key in checkpoint.keys()
    ) and any(
        torch.is_tensor(value)
        for value in checkpoint.values()
    ):
        print("Checkpoint itself is a direct state dictionary.")

        return checkpoint

    raise KeyError(
        "Could not find a model state dictionary "
        "inside the checkpoint."
    )


def remove_module_prefix(
    state_dict,
):
    cleaned_state_dict = OrderedDict()

    for key, value in state_dict.items():
        cleaned_key = (
            key[7:]
            if key.startswith("module.")
            else key
        )

        cleaned_state_dict[
            cleaned_key
        ] = value

    return cleaned_state_dict


def print_checkpoint_metadata(
    checkpoint,
):
    if not isinstance(checkpoint, dict):
        return

    metadata_keys = [
        "epoch",
        "stage",
        "score",
        "best_score",
        "val_score",
        "miou",
        "mIoU",
        "loss",
    ]

    print("\nCheckpoint metadata:")

    metadata_found = False

    for key in metadata_keys:
        if key in checkpoint:
            value = checkpoint[key]

            if (
                isinstance(value, (str, int, float, bool))
                or value is None
            ):
                print(f"  {key}: {value}")
                metadata_found = True

    if not metadata_found:
        print("  No simple metadata fields found.")


def load_best_siamese_damage_model(
    checkpoint_path,
):
    if "SiameseUNet" not in globals():
        raise NameError(
            "SiameseUNet architecture is not defined. "
            "Run the model architecture cell first."
        )

    checkpoint_path = Path(
        checkpoint_path
    )

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found:\n{checkpoint_path}"
        )

    model = SiameseUNet(
        in_channels=3,
        building_classes=2,
        damage_classes=4,
        init_features=16,
    ).to(device)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    print_checkpoint_metadata(
        checkpoint
    )

    state_dict = extract_model_state_dict(
        checkpoint
    )

    state_dict = remove_module_prefix(
        state_dict
    )

    incompatible = model.load_state_dict(
        state_dict,
        strict=True,
    )

    print("\nCheckpoint loading result:")
    print("Missing keys   :", incompatible.missing_keys)
    print("Unexpected keys:", incompatible.unexpected_keys)

    model.eval()

    print("\n✅ Best Siamese damage model loaded.")
    print("Checkpoint:", checkpoint_path)
    print("Device    :", DEVICE)

    return model, checkpoint


siamese_test_model, siamese_checkpoint = (
    load_best_siamese_damage_model(
        checkpoint_path=BEST_DAMAGE_CHECKPOINT
    )
)


Checkpoint metadata:
  epoch: 41
  stage: damage
  best_score: 0.13759076595306396
Model state dictionary found in checkpoint['model'].

Checkpoint loading result:
Missing keys   : []
Unexpected keys: []

✅ Best Siamese damage model loaded.
Checkpoint: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/models/damage/best_damage_model.pth
Device    : cuda


In [32]:
# ============================================================
# STEP 4.4 — ONE-BATCH PREDICTION SANITY CHECK
# ============================================================

import torch
import numpy as np


def apply_official_pre_building_gate(
    pre_building_logits,
    damage_logits,
):
    """
    Microsoft Siamese U-Net inference logic:

    1. Predict building localization from the pre-event image.
    2. Restrict damage prediction to predicted buildings.
    3. Outside predicted buildings, all damage logits become zero;
       argmax therefore returns class 0 (background).
    """
    predicted_pre_building = torch.argmax(
        pre_building_logits,
        dim=1,
    )

    predicted_pre_building = (
        predicted_pre_building == 1
    )

    gated_damage_logits = (
        damage_logits
        * predicted_pre_building.unsqueeze(1).to(
            damage_logits.dtype
        )
    )

    return (
        gated_damage_logits,
        predicted_pre_building,
    )


test_batch = next(
    iter(test_loader)
)

(
    pre_images,
    post_images,
    damage_targets,
    building_masks,
    event_names,
    file_paths,
) = test_batch

pre_images_gpu = pre_images.to(
    DEVICE,
    non_blocking=True,
)

post_images_gpu = post_images.to(
    DEVICE,
    non_blocking=True,
)

siamese_test_model.eval()

with torch.inference_mode():
    (
        pre_building_logits,
        post_building_logits,
        raw_damage_logits,
    ) = siamese_test_model(
        pre_images_gpu,
        post_images_gpu,
    )

    (
        gated_damage_logits,
        predicted_pre_building,
    ) = apply_official_pre_building_gate(
        pre_building_logits=
            pre_building_logits,

        damage_logits=
            raw_damage_logits,
    )

    damage_predictions = torch.argmax(
        gated_damage_logits,
        dim=1,
    )

print("=" * 72)
print("ONE-BATCH SIAMESE PREDICTION CHECK")
print("=" * 72)
print("Pre images             :", tuple(pre_images.shape))
print("Post images            :", tuple(post_images.shape))
print("Damage targets         :", tuple(damage_targets.shape))
print("Building masks         :", tuple(building_masks.shape))
print("Pre-building logits    :", tuple(pre_building_logits.shape))
print("Post-building logits   :", tuple(post_building_logits.shape))
print("Raw damage logits      :", tuple(raw_damage_logits.shape))
print("Gated damage logits    :", tuple(gated_damage_logits.shape))
print("Damage predictions     :", tuple(damage_predictions.shape))
print(
    "Target classes         :",
    torch.unique(damage_targets).tolist(),
)
print(
    "Predicted classes      :",
    torch.unique(damage_predictions).tolist(),
)
print(
    "Predicted building rate:",
    f"{predicted_pre_building.float().mean().item():.4f}",
)
print("First event            :", event_names[0])
print("First file             :", file_paths[0])
print("=" * 72)

assert pre_images.shape == post_images.shape
assert pre_building_logits.shape[1] == 2
assert post_building_logits.shape[1] == 2
assert raw_damage_logits.shape[1] == NUM_CLASSES
assert damage_predictions.shape == damage_targets.shape
assert torch.isfinite(raw_damage_logits).all()
assert torch.isfinite(gated_damage_logits).all()

valid_prediction_classes = set(
    torch.unique(
        damage_predictions
    ).detach().cpu().tolist()
)

assert valid_prediction_classes.issubset(
    {0, 1, 2, 3}
)

print("\n✅ Siamese U-Net test prediction sanity check passed.")

ONE-BATCH SIAMESE PREDICTION CHECK
Pre images             : (16, 3, 256, 256)
Post images            : (16, 3, 256, 256)
Damage targets         : (16, 256, 256)
Building masks         : (16, 256, 256)
Pre-building logits    : (16, 2, 256, 256)
Post-building logits   : (16, 2, 256, 256)
Raw damage logits      : (16, 4, 256, 256)
Gated damage logits    : (16, 4, 256, 256)
Damage predictions     : (16, 256, 256)
Target classes         : [0, 1, 2, 3]
Predicted classes      : [0, 1, 2, 3]
Predicted building rate: 0.4441
First event            : HURRICANE-DORIAN
First file             : /content/ebd_HURRICANE-DORIAN_local/HURRICANE-DORIAN_prepost_npz_512valid_as_256/HURRICANE-DORIAN_000011_0_0.npz

✅ Siamese U-Net test prediction sanity check passed.


In [33]:
# ============================================================
# STEP 5 — COMPLETE QUANTITATIVE TEST EVALUATION
# ============================================================

import os
import json
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm.auto import tqdm
from scipy import ndimage
from matplotlib import font_manager as fm
from matplotlib import cm as mpl_cm
from matplotlib import colors as mpl_colors


# ============================================================
# 1. OUTPUT PATHS
# ============================================================

TEST_OUTPUT_DIR = Path(
    SIAMESE_RUN_ROOT /
    f"{EVENT_TAG}_test_metrics"
)

TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

EXCEL_PATH = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_Siamese_UNet_test_metrics.xlsx"
)

SUMMARY_TEXT_PATH = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_Siamese_UNet_test_summary.txt"
)

SUMMARY_JSON_PATH = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_Siamese_UNet_test_summary.json"
)

LEGACY_PIXEL_VIOLIN_PNG = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_pixelwise_patch_violin.png"
)

LEGACY_PIXEL_VIOLIN_PDF = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_pixelwise_patch_violin.pdf"
)

LEGACY_BUILDING_VIOLIN_PNG = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_buildingwise_patch_violin.png"
)

LEGACY_BUILDING_VIOLIN_PDF = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_buildingwise_patch_violin.pdf"
)

STRICT_PIXEL_VIOLIN_PNG = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_strict_pixelwise_patch_violin.png"
)

STRICT_PIXEL_VIOLIN_PDF = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_strict_pixelwise_patch_violin.pdf"
)

STRICT_BUILDING_VIOLIN_PNG = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_strict_buildingwise_patch_violin.png"
)

STRICT_BUILDING_VIOLIN_PDF = (
    TEST_OUTPUT_DIR /
    f"{EVENT_TAG}_strict_buildingwise_patch_violin.pdf"
)

FIGURE_DPI = 800


# ============================================================
# 2. FILENAME PARSING
# ============================================================

def parse_test_patch_name(path):
    """
    Expected:
      HURRICANE-DORIAN_000011_0_0.npz
      HURRICANE-DORIAN_000011_0_256.npz
    """
    stem = Path(path).stem

    try:
        base_name, y_string, x_string = stem.rsplit(
            "_",
            2,
        )

        return (
            base_name,
            int(y_string),
            int(x_string),
        )

    except Exception:
        return stem, np.nan, np.nan


# ============================================================
# 3. CONFUSION MATRIX
# ============================================================

def confusion_matrix_numpy(
    prediction,
    target,
    valid_mask,
    num_classes=4,
):
    prediction = np.asarray(
        prediction
    ).reshape(-1)

    target = np.asarray(
        target
    ).reshape(-1)

    valid_mask = np.asarray(
        valid_mask
    ).reshape(-1).astype(bool)

    valid = (
        valid_mask
        & (target >= 0)
        & (target < num_classes)
        & (prediction >= 0)
        & (prediction < num_classes)
    )

    prediction = prediction[
        valid
    ].astype(np.int64)

    target = target[
        valid
    ].astype(np.int64)

    if target.size == 0:
        return np.zeros(
            (num_classes, num_classes),
            dtype=np.float64,
        )

    matrix = np.bincount(
        num_classes * target + prediction,
        minlength=num_classes ** 2,
    ).reshape(
        num_classes,
        num_classes,
    )

    return matrix.astype(
        np.float64
    )


# ============================================================
# 4. LEGACY STRATA METRICS
# ============================================================

def compute_legacy_strata_metrics(
    confusion_matrix,
):
    """
    Exact structure of the previous STRATA evaluation:

        cm_eval = cm[1:, 1:]

    Important:
    Predictions assigned to background on true damage pixels
    are excluded by this calculation.
    """
    matrix = np.asarray(
        confusion_matrix,
        dtype=np.float64,
    )

    matrix_eval = matrix[1:, 1:]

    true_positive = np.diag(
        matrix_eval
    )

    false_positive = (
        matrix_eval.sum(axis=0)
        - true_positive
    )

    false_negative = (
        matrix_eval.sum(axis=1)
        - true_positive
    )

    total = matrix_eval.sum()

    accuracy = (
        true_positive.sum()
        / (total + 1e-6)
    )

    expected_agreement = (
        (
            matrix_eval.sum(axis=0)
            * matrix_eval.sum(axis=1)
        ).sum()
        / ((total ** 2) + 1e-6)
    )

    kappa = (
        (accuracy - expected_agreement)
        / (1.0 - expected_agreement + 1e-6)
    )

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
            + 1e-6
        )
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
            + 1e-6
        )
    )

    f1 = (
        2.0 * precision * recall
        / (precision + recall + 1e-6)
    )

    iou = (
        true_positive
        / (
            true_positive
            + false_positive
            + false_negative
            + 1e-6
        )
    )

    metrics = {
        "accuracy": float(accuracy),
        "precision_macro": float(np.nanmean(precision)),
        "recall_macro": float(np.nanmean(recall)),
        "f1_macro": float(np.nanmean(f1)),
        "kappa": float(kappa),
        "miou_3class": float(np.nanmean(iou)),
        "miou_rare_2class": float(np.nanmean(iou[1:3])),
    }

    for class_index, class_name in enumerate(
        FG_CLASS_NAMES
    ):
        metrics[
            f"precision_{class_name}"
        ] = float(precision[class_index])

        metrics[
            f"recall_{class_name}"
        ] = float(recall[class_index])

        metrics[
            f"f1_{class_name}"
        ] = float(f1[class_index])

        metrics[
            f"iou_{class_name}"
        ] = float(iou[class_index])

    return metrics


# ============================================================
# 5. STRICT METRICS
# ============================================================

def compute_strict_metrics(
    confusion_matrix,
):
    """
    Correct foreground evaluation:

    - Ground-truth classes: 1, 2, 3
    - Predictions can be: 0, 1, 2, 3
    - Predicting background for a foreground pixel is a FN.
    """
    matrix = np.asarray(
        confusion_matrix,
        dtype=np.float64,
    ).copy()

    foreground_classes = [
        1,
        2,
        3,
    ]

    true_positive = np.array([
        matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    false_positive = np.array([
        matrix[:, class_id].sum()
        - matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    false_negative = np.array([
        matrix[class_id, :].sum()
        - matrix[class_id, class_id]
        for class_id in foreground_classes
    ])

    # True-background pixels are not included in strict damage accuracy.
    strict_matrix = matrix.copy()
    strict_matrix[0, :] = 0.0

    total = strict_matrix.sum()

    accuracy = (
        true_positive.sum()
        / (total + 1e-6)
    )

    row_marginals = strict_matrix.sum(
        axis=1
    )

    column_marginals = strict_matrix.sum(
        axis=0
    )

    expected_agreement = (
        np.sum(
            row_marginals
            * column_marginals
        )
        / ((total ** 2) + 1e-6)
    )

    kappa = (
        (accuracy - expected_agreement)
        / (1.0 - expected_agreement + 1e-6)
    )

    precision = (
        true_positive
        / (
            true_positive
            + false_positive
            + 1e-6
        )
    )

    recall = (
        true_positive
        / (
            true_positive
            + false_negative
            + 1e-6
        )
    )

    f1 = (
        2.0 * precision * recall
        / (precision + recall + 1e-6)
    )

    iou = (
        true_positive
        / (
            true_positive
            + false_positive
            + false_negative
            + 1e-6
        )
    )

    metrics = {
        "accuracy": float(accuracy),
        "precision_macro": float(np.nanmean(precision)),
        "recall_macro": float(np.nanmean(recall)),
        "f1_macro": float(np.nanmean(f1)),
        "kappa": float(kappa),
        "miou_3class": float(np.nanmean(iou)),
        "miou_rare_2class": float(np.nanmean(iou[1:3])),
    }

    for class_index, class_name in enumerate(
        FG_CLASS_NAMES
    ):
        metrics[
            f"precision_{class_name}"
        ] = float(precision[class_index])

        metrics[
            f"recall_{class_name}"
        ] = float(recall[class_index])

        metrics[
            f"f1_{class_name}"
        ] = float(f1[class_index])

        metrics[
            f"iou_{class_name}"
        ] = float(iou[class_index])

    return metrics


# ============================================================
# 6. BUILDING-MAJORITY MAP
# ============================================================

def building_majority_map(
    label_map,
    building_mask,
):
    """
    Same connected-component majority procedure used in
    the previous STRATA independent-event evaluation.
    """
    building_binary = (
        np.asarray(building_mask) > 0
    )

    connected_labels, number_of_regions = (
        ndimage.label(
            building_binary
        )
    )

    output = np.zeros_like(
        label_map,
        dtype=np.uint8,
    )

    for region_id in range(
        1,
        number_of_regions + 1,
    ):
        region = (
            connected_labels == region_id
        )

        class_values = np.asarray(
            label_map
        )[region]

        class_values = class_values[
            class_values > 0
        ]

        if class_values.size == 0:
            majority_class = 0

        else:
            majority_class = np.bincount(
                class_values.astype(np.int64),
                minlength=NUM_CLASSES,
            ).argmax()

        output[
            region
        ] = majority_class

    return output


# ============================================================
# 7. ONE DATAFRAME ROW
# ============================================================

def make_patch_metric_row(
    file_path,
    event_name,
    valid_building_pixels,
    evaluation_mode,
    metrics,
):
    base_name, y, x = parse_test_patch_name(
        file_path
    )

    row = {
        "file": Path(file_path).name,
        "base_name": base_name,
        "y": y,
        "x": x,
        "event": event_name,
        "valid_building_pixels": int(
            valid_building_pixels
        ),
        "mode": evaluation_mode,
    }

    row.update(
        metrics
    )

    return row


# ============================================================
# 8. COMPLETE MODEL EVALUATION
# ============================================================

def evaluate_siamese_dorian_test(
    model,
    loader,
):
    pixel_confusion_global = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.float64,
    )

    building_confusion_global = np.zeros(
        (NUM_CLASSES, NUM_CLASSES),
        dtype=np.float64,
    )

    legacy_pixel_rows = []
    legacy_building_rows = []

    strict_pixel_rows = []
    strict_building_rows = []

    predicted_class_pixel_counts = Counter()
    target_class_pixel_counts = Counter()

    model.eval()

    with torch.inference_mode():
        for batch in tqdm(
            loader,
            desc=f"Evaluating {EVENT_NAME}",
            unit="batch",
        ):
            (
                pre_images,
                post_images,
                damage_targets,
                building_masks,
                event_names,
                file_paths,
            ) = batch

            pre_images_gpu = pre_images.to(
                DEVICE,
                non_blocking=True,
            )

            post_images_gpu = post_images.to(
                DEVICE,
                non_blocking=True,
            )

            (
                pre_building_logits,
                post_building_logits,
                raw_damage_logits,
            ) = model(
                pre_images_gpu,
                post_images_gpu,
            )

            (
                gated_damage_logits,
                predicted_pre_buildings,
            ) = apply_official_pre_building_gate(
                pre_building_logits=
                    pre_building_logits,

                damage_logits=
                    raw_damage_logits,
            )

            predictions = torch.argmax(
                gated_damage_logits,
                dim=1,
            )

            predictions_numpy = (
                predictions
                .detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            targets_numpy = (
                damage_targets
                .detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            buildings_numpy = (
                building_masks
                .detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
            )

            for batch_index in range(
                predictions_numpy.shape[0]
            ):
                prediction = predictions_numpy[
                    batch_index
                ]

                target = targets_numpy[
                    batch_index
                ]

                building = buildings_numpy[
                    batch_index
                ]

                event_name = event_names[
                    batch_index
                ]

                file_path = file_paths[
                    batch_index
                ]

                valid_building = (
                    building > 0
                )

                # Official model prediction is retained.
                # GT building mask is used only for evaluation.
                for class_id, count in zip(
                    *np.unique(
                        prediction[valid_building],
                        return_counts=True,
                    )
                ):
                    predicted_class_pixel_counts[
                        int(class_id)
                    ] += int(count)

                for class_id, count in zip(
                    *np.unique(
                        target[valid_building],
                        return_counts=True,
                    )
                ):
                    target_class_pixel_counts[
                        int(class_id)
                    ] += int(count)

                # --------------------------------------------
                # PIXEL-WISE CONFUSION MATRIX
                # --------------------------------------------

                pixel_confusion_patch = (
                    confusion_matrix_numpy(
                        prediction=prediction,
                        target=target,
                        valid_mask=valid_building,
                        num_classes=NUM_CLASSES,
                    )
                )

                pixel_confusion_global += (
                    pixel_confusion_patch
                )

                legacy_pixel_metrics = (
                    compute_legacy_strata_metrics(
                        pixel_confusion_patch
                    )
                )

                strict_pixel_metrics = (
                    compute_strict_metrics(
                        pixel_confusion_patch
                    )
                )

                legacy_pixel_rows.append(
                    make_patch_metric_row(
                        file_path=file_path,
                        event_name=event_name,
                        valid_building_pixels=
                            valid_building.sum(),
                        evaluation_mode=
                            "pixel_wise",
                        metrics=
                            legacy_pixel_metrics,
                    )
                )

                strict_pixel_rows.append(
                    make_patch_metric_row(
                        file_path=file_path,
                        event_name=event_name,
                        valid_building_pixels=
                            valid_building.sum(),
                        evaluation_mode=
                            "strict_pixel_wise",
                        metrics=
                            strict_pixel_metrics,
                    )
                )

                # --------------------------------------------
                # BUILDING-MAJORITY CONFUSION MATRIX
                # --------------------------------------------

                target_building_majority = (
                    building_majority_map(
                        label_map=target,
                        building_mask=building,
                    )
                )

                prediction_building_majority = (
                    building_majority_map(
                        label_map=prediction,
                        building_mask=building,
                    )
                )

                building_confusion_patch = (
                    confusion_matrix_numpy(
                        prediction=
                            prediction_building_majority,

                        target=
                            target_building_majority,

                        valid_mask=
                            valid_building,

                        num_classes=
                            NUM_CLASSES,
                    )
                )

                building_confusion_global += (
                    building_confusion_patch
                )

                legacy_building_metrics = (
                    compute_legacy_strata_metrics(
                        building_confusion_patch
                    )
                )

                strict_building_metrics = (
                    compute_strict_metrics(
                        building_confusion_patch
                    )
                )

                legacy_building_rows.append(
                    make_patch_metric_row(
                        file_path=file_path,
                        event_name=event_name,
                        valid_building_pixels=
                            valid_building.sum(),
                        evaluation_mode=
                            "building_wise_majority",
                        metrics=
                            legacy_building_metrics,
                    )
                )

                strict_building_rows.append(
                    make_patch_metric_row(
                        file_path=file_path,
                        event_name=event_name,
                        valid_building_pixels=
                            valid_building.sum(),
                        evaluation_mode=
                            "strict_building_majority",
                        metrics=
                            strict_building_metrics,
                    )
                )

    # --------------------------------------------------------
    # GLOBAL METRICS
    # --------------------------------------------------------

    legacy_global_pixel = (
        compute_legacy_strata_metrics(
            pixel_confusion_global
        )
    )

    legacy_global_building = (
        compute_legacy_strata_metrics(
            building_confusion_global
        )
    )

    strict_global_pixel = (
        compute_strict_metrics(
            pixel_confusion_global
        )
    )

    strict_global_building = (
        compute_strict_metrics(
            building_confusion_global
        )
    )

    results = {
        "pixel_confusion":
            pixel_confusion_global,

        "building_confusion":
            building_confusion_global,

        "legacy_patch_pixel_df":
            pd.DataFrame(legacy_pixel_rows),

        "legacy_patch_building_df":
            pd.DataFrame(legacy_building_rows),

        "strict_patch_pixel_df":
            pd.DataFrame(strict_pixel_rows),

        "strict_patch_building_df":
            pd.DataFrame(strict_building_rows),

        "legacy_global_pixel":
            legacy_global_pixel,

        "legacy_global_building":
            legacy_global_building,

        "strict_global_pixel":
            strict_global_pixel,

        "strict_global_building":
            strict_global_building,

        "predicted_class_counts":
            dict(
                sorted(
                    predicted_class_pixel_counts.items()
                )
            ),

        "target_class_counts":
            dict(
                sorted(
                    target_class_pixel_counts.items()
                )
            ),
    }

    return results


# ============================================================
# 9. RUN COMPLETE EVALUATION
# ============================================================

test_results = evaluate_siamese_dorian_test(
    model=siamese_test_model,
    loader=test_loader,
)

print("\n✅ Full Dorian prediction completed.")

Evaluating HURRICANE-DORIAN:   0%|          | 0/46 [00:00<?, ?batch/s]


✅ Full Dorian prediction completed.


In [34]:
# ============================================================
# STEP 5.1 — SAVE EXCEL AND SUMMARY FILES
# ============================================================

def metrics_to_global_dataframe(
    metrics,
    mode,
    protocol,
):
    return pd.DataFrame([{
        "event": EVENT_NAME,
        "model": "Siamese_UNet",
        "run": "budget500_seed42",
        "mode": mode,
        "protocol": protocol,
        **metrics,
    }])


legacy_global_pixel_df = metrics_to_global_dataframe(
    test_results["legacy_global_pixel"],
    mode="pixel_wise_global",
    protocol="legacy_STRATA",
)

legacy_global_building_df = metrics_to_global_dataframe(
    test_results["legacy_global_building"],
    mode="building_wise_majority_global",
    protocol="legacy_STRATA",
)

strict_global_pixel_df = metrics_to_global_dataframe(
    test_results["strict_global_pixel"],
    mode="pixel_wise_global",
    protocol="strict",
)

strict_global_building_df = metrics_to_global_dataframe(
    test_results["strict_global_building"],
    mode="building_wise_majority_global",
    protocol="strict",
)


pixel_confusion_df = pd.DataFrame(
    test_results["pixel_confusion"],
    index=[
        f"true_{name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"pred_{name}"
        for name in CLASS_NAMES
    ],
)

building_confusion_df = pd.DataFrame(
    test_results["building_confusion"],
    index=[
        f"true_{name}"
        for name in CLASS_NAMES
    ],
    columns=[
        f"pred_{name}"
        for name in CLASS_NAMES
    ],
)


configuration_df = pd.DataFrame([
    {
        "parameter": "event",
        "value": EVENT_NAME,
    },
    {
        "parameter": "model",
        "value": "Siamese_UNet",
    },
    {
        "parameter": "run",
        "value": "budget500_seed42",
    },
    {
        "parameter": "test_patches",
        "value": len(test_dataset),
    },
    {
        "parameter": "complete_512_blocks",
        "value": len(test_dataset) // 4,
    },
    {
        "parameter": "checkpoint",
        "value": str(BEST_DAMAGE_CHECKPOINT),
    },
    {
        "parameter": "test_data",
        "value": str(PAIRED_TEST_ROOT),
    },
    {
        "parameter": "official_pre_building_gate",
        "value": True,
    },
])


with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl",
) as writer:

    # Same six primary sheets as previous STRATA code
    legacy_global_pixel_df.to_excel(
        writer,
        sheet_name="global_pixel_metrics",
        index=False,
    )

    legacy_global_building_df.to_excel(
        writer,
        sheet_name="global_building_metrics",
        index=False,
    )

    test_results[
        "legacy_patch_pixel_df"
    ].to_excel(
        writer,
        sheet_name="patch_pixel_metrics",
        index=False,
    )

    test_results[
        "legacy_patch_building_df"
    ].to_excel(
        writer,
        sheet_name="patch_building_metrics",
        index=False,
    )

    pixel_confusion_df.to_excel(
        writer,
        sheet_name="cm_pixel",
    )

    building_confusion_df.to_excel(
        writer,
        sheet_name="cm_building",
    )

    # Additional corrected metrics
    strict_global_pixel_df.to_excel(
        writer,
        sheet_name="strict_global_pixel",
        index=False,
    )

    strict_global_building_df.to_excel(
        writer,
        sheet_name="strict_global_building",
        index=False,
    )

    test_results[
        "strict_patch_pixel_df"
    ].to_excel(
        writer,
        sheet_name="strict_patch_pixel",
        index=False,
    )

    test_results[
        "strict_patch_building_df"
    ].to_excel(
        writer,
        sheet_name="strict_patch_building",
        index=False,
    )

    configuration_df.to_excel(
        writer,
        sheet_name="configuration",
        index=False,
    )


summary_dictionary = {
    "event": EVENT_NAME,
    "model": "Siamese_UNet",
    "run": "budget500_seed42",
    "checkpoint": str(BEST_DAMAGE_CHECKPOINT),
    "test_patches": len(test_dataset),
    "complete_512_blocks": len(test_dataset) // 4,
    "legacy_global_pixel":
        test_results["legacy_global_pixel"],
    "legacy_global_building":
        test_results["legacy_global_building"],
    "strict_global_pixel":
        test_results["strict_global_pixel"],
    "strict_global_building":
        test_results["strict_global_building"],
    "predicted_class_counts":
        test_results["predicted_class_counts"],
    "target_class_counts":
        test_results["target_class_counts"],
}


with open(
    SUMMARY_JSON_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        summary_dictionary,
        output_file,
        indent=4,
        ensure_ascii=False,
    )


with open(
    SUMMARY_TEXT_PATH,
    "w",
    encoding="utf-8",
) as output_file:

    output_file.write(
        "Siamese U-Net Independent Event Test\n"
    )

    output_file.write(
        "=" * 60 + "\n"
    )

    output_file.write(
        f"Event: {EVENT_NAME}\n"
    )

    output_file.write(
        "Model: Siamese_UNet\n"
    )

    output_file.write(
        "Run: budget500_seed42\n"
    )

    output_file.write(
        f"Checkpoint: {BEST_DAMAGE_CHECKPOINT}\n"
    )

    output_file.write(
        f"Test patches: {len(test_dataset)}\n"
    )

    output_file.write(
        f"Complete 512 blocks: {len(test_dataset) // 4}\n\n"
    )

    for section_name in [
        "legacy_global_pixel",
        "legacy_global_building",
        "strict_global_pixel",
        "strict_global_building",
    ]:
        output_file.write(
            section_name.upper() + "\n"
        )

        for metric_name, metric_value in (
            summary_dictionary[
                section_name
            ].items()
        ):
            output_file.write(
                f"{metric_name}: "
                f"{metric_value:.6f}\n"
            )

        output_file.write("\n")


print("=" * 72)
print("QUANTITATIVE RESULTS SAVED")
print("=" * 72)
print("Excel :", EXCEL_PATH)
print("Text  :", SUMMARY_TEXT_PATH)
print("JSON  :", SUMMARY_JSON_PATH)
print("=" * 72)

QUANTITATIVE RESULTS SAVED
Excel : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_Siamese_UNet_test_metrics.xlsx
Text  : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_Siamese_UNet_test_summary.txt
JSON  : /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_Siamese_UNet_test_summary.json


In [35]:
# ============================================================
# STEP 5.2 — SAVE QUANTITATIVE VIOLIN CHARTS
# ============================================================

def get_available_serif_font():
    available_fonts = {
        font.name
        for font in fm.fontManager.ttflist
    }

    preferred_fonts = [
        "Times New Roman",
        "Liberation Serif",
        "Nimbus Roman",
        "DejaVu Serif",
    ]

    for font_name in preferred_fonts:
        if font_name in available_fonts:
            return font_name

    return "serif"


def save_violin_plot(
    dataframe,
    output_png,
    output_pdf,
    mode_name,
):
    plot_dataframe = dataframe.copy()

    plotted_metrics = [
        ("accuracy", "Accuracy"),
        ("f1_macro", "F-score"),
        ("miou_3class", "mIoU"),
    ]

    violin_data = []
    labels = []

    for column_name, display_name in plotted_metrics:
        if column_name not in plot_dataframe.columns:
            raise KeyError(
                f"Column not found: {column_name}"
            )

        values = pd.to_numeric(
            plot_dataframe[column_name],
            errors="coerce",
        ).dropna().to_numpy()

        values = values[
            np.isfinite(values)
        ]

        if len(values) == 0:
            raise ValueError(
                f"No valid values for {column_name}."
            )

        if len(values) == 1:
            values = np.repeat(
                values,
                2,
            )

        violin_data.append(
            values
        )

        labels.append(
            display_name
        )

    selected_font = (
        get_available_serif_font()
    )

    print(
        f"Font for {mode_name}:",
        selected_font,
    )

    plt.rcParams["font.family"] = selected_font
    plt.rcParams["font.size"] = 12
    plt.rcParams["axes.linewidth"] = 1.0
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    figure, axis = plt.subplots(
        figsize=(6.4, 4.4)
    )

    violin_parts = axis.violinplot(
        violin_data,
        showmeans=False,
        showmedians=False,
        showextrema=False,
        widths=0.78,
    )

    color_map = mpl_cm.get_cmap(
        "jet"
    )

    normalization = mpl_colors.Normalize(
        vmin=0.0,
        vmax=1.0,
    )

    for violin_index, body in enumerate(
        violin_parts["bodies"]
    ):
        mean_value = float(
            np.nanmean(
                violin_data[violin_index]
            )
        )

        body.set_facecolor(
            color_map(
                normalization(mean_value)
            )
        )

        body.set_edgecolor(
            "black"
        )

        body.set_alpha(
            0.45
        )

        body.set_linewidth(
            1.1
        )

    random_generator = np.random.default_rng(
        42
    )

    for position, values in enumerate(
        violin_data,
        start=1,
    ):
        jittered_x = random_generator.normal(
            loc=position,
            scale=0.045,
            size=len(values),
        )

        axis.scatter(
            jittered_x,
            values,
            c=values,
            cmap="jet",
            vmin=0.0,
            vmax=1.0,
            s=13,
            alpha=0.72,
            edgecolors="none",
            zorder=3,
        )

        median_value = np.nanmedian(
            values
        )

        axis.plot(
            [
                position - 0.22,
                position + 0.22,
            ],
            [
                median_value,
                median_value,
            ],
            color="black",
            linewidth=1.4,
            zorder=4,
        )

        mean_value = np.nanmean(
            values
        )

        axis.scatter(
            position,
            mean_value,
            marker="D",
            s=38,
            color="black",
            zorder=5,
        )

    axis.set_xticks(
        np.arange(
            1,
            len(labels) + 1,
        )
    )

    axis.set_xticklabels(
        labels
    )

    axis.set_ylabel(
        "Patch-level score"
    )

    axis.set_ylim(
        0.0,
        1.02,
    )

    axis.grid(
        axis="y",
        linestyle="--",
        linewidth=0.5,
        alpha=0.45,
    )

    axis.set_title(
        ""
    )

    axis.spines[
        "top"
    ].set_visible(False)

    axis.spines[
        "right"
    ].set_visible(False)

    scalar_mappable = mpl_cm.ScalarMappable(
        cmap="jet",
        norm=normalization,
    )

    scalar_mappable.set_array([])

    color_bar = figure.colorbar(
        scalar_mappable,
        ax=axis,
        fraction=0.045,
        pad=0.035,
    )

    color_bar.set_label(
        "Score value"
    )

    color_bar.outline.set_linewidth(
        0.8
    )

    figure.tight_layout()

    figure.savefig(
        output_png,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    figure.savefig(
        output_pdf,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    print("Saved:", output_png)
    print("Saved:", output_pdf)


# Legacy plots — directly comparable with previous STRATA plots
save_violin_plot(
    dataframe=
        test_results["legacy_patch_pixel_df"],

    output_png=
        LEGACY_PIXEL_VIOLIN_PNG,

    output_pdf=
        LEGACY_PIXEL_VIOLIN_PDF,

    mode_name=
        "legacy pixel-wise",
)

save_violin_plot(
    dataframe=
        test_results["legacy_patch_building_df"],

    output_png=
        LEGACY_BUILDING_VIOLIN_PNG,

    output_pdf=
        LEGACY_BUILDING_VIOLIN_PDF,

    mode_name=
        "legacy building-wise",
)


# Strict plots — scientifically corrected protocol
save_violin_plot(
    dataframe=
        test_results["strict_patch_pixel_df"],

    output_png=
        STRICT_PIXEL_VIOLIN_PNG,

    output_pdf=
        STRICT_PIXEL_VIOLIN_PDF,

    mode_name=
        "strict pixel-wise",
)

save_violin_plot(
    dataframe=
        test_results["strict_patch_building_df"],

    output_png=
        STRICT_BUILDING_VIOLIN_PNG,

    output_pdf=
        STRICT_BUILDING_VIOLIN_PDF,

    mode_name=
        "strict building-wise",
)

print("\n✅ All quantitative charts were saved.")

Font for legacy pixel-wise: Liberation Serif


/tmp/ipykernel_9841/4210202715.py:103: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(


Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_pixelwise_patch_violin.png
Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_pixelwise_patch_violin.pdf
Font for legacy building-wise: Liberation Serif


/tmp/ipykernel_9841/4210202715.py:103: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(


Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_buildingwise_patch_violin.png
Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_buildingwise_patch_violin.pdf
Font for strict pixel-wise: Liberation Serif


/tmp/ipykernel_9841/4210202715.py:103: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(


Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_strict_pixelwise_patch_violin.png
Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_strict_pixelwise_patch_violin.pdf
Font for strict building-wise: Liberation Serif


/tmp/ipykernel_9841/4210202715.py:103: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  color_map = mpl_cm.get_cmap(


Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_strict_buildingwise_patch_violin.png
Saved: /content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics/HURRICANE_DORIAN_strict_buildingwise_patch_violin.pdf

✅ All quantitative charts were saved.


In [36]:
# ============================================================
# STEP 5.3 — DISPLAY GLOBAL RESULTS
# ============================================================

comparison_rows = []

for protocol_name, pixel_key, building_key in [
    (
        "Legacy_STRATA",
        "legacy_global_pixel",
        "legacy_global_building",
    ),
    (
        "Strict",
        "strict_global_pixel",
        "strict_global_building",
    ),
]:
    for mode_name, result_key in [
        ("Pixel-wise", pixel_key),
        ("Building-majority", building_key),
    ]:
        metrics = test_results[result_key]

        comparison_rows.append({
            "Protocol": protocol_name,
            "Mode": mode_name,
            "Accuracy": metrics["accuracy"],
            "Macro_F1": metrics["f1_macro"],
            "Kappa": metrics["kappa"],
            "mIoU_3class": metrics["miou_3class"],
            "mIoU_rare": metrics["miou_rare_2class"],
            "IoU_Intact": metrics["iou_intact"],
            "IoU_Damage": metrics["iou_damage"],
            "IoU_Destroyed": metrics["iou_destroyed"],
        })

global_comparison_df = pd.DataFrame(
    comparison_rows
)

display(
    global_comparison_df.round(4)
)

print("\nTarget class pixel counts:")
print(
    test_results["target_class_counts"]
)

print("\nPredicted class pixel counts:")
print(
    test_results["predicted_class_counts"]
)

print("\nOutput directory:")
print(TEST_OUTPUT_DIR)

,Protocol,Mode,Accuracy,Macro_F1,Kappa,mIoU_3class,mIoU_rare,IoU_Intact,IoU_Damage,IoU_Destroyed
0,Legacy_STRATA,Pixel-wise,0.5680,0.5292,0.2941,0.3651,0.3265,0.4424,0.4035,0.2494
1,Legacy_STRATA,Building-majority,0.5814,0.5369,0.3139,0.3734,0.3329,0.4545,0.4213,0.2445
2,Strict,Pixel-wise,0.5569,0.5234,0.2855,0.3599,0.3204,0.4388,0.3971,0.2437
3,Strict,Building-majority,0.5802,0.5362,0.3131,0.3728,0.3323,0.4537,0.4212,0.2434



Target class pixel counts:
{1: 1589808, 2: 1961265, 3: 823356}

Predicted class pixel counts:
{0: 85355, 1: 2059041, 2: 1898692, 3: 331341}

Output directory:
/content/drive/MyDrive/STRATA_runs/PrePost_Baselines/Siamese_UNet/budget500_seed42/HURRICANE_DORIAN_test_metrics
